# Citiz France — carte interactive et recalcul Python
Python natif, sans Spark. Fichiers CSV. Vue nationale des territoires et détail 200 m par département.


In [ ]:
import sys, subprocess, importlib.util
names=["pandas","numpy","scipy","pyproj","folium","ipywidgets"]
missing=[n for n in names if importlib.util.find_spec(n) is None]
if missing: subprocess.check_call([sys.executable,"-m","pip","install",*missing])


In [ ]:
"""Journaux techniques hors de l'interface ; les exceptions restent visibles."""
from contextlib import contextmanager, redirect_stdout, redirect_stderr
from datetime import datetime, timezone
from pathlib import Path
import traceback


@contextmanager
def diagnostic_output(workdir, stage):
    folder=Path(workdir)/'diagnostics'
    folder.mkdir(parents=True,exist_ok=True)
    path=folder/(stage+'.log')
    with path.open('w',encoding='utf-8') as stream:
        stream.write(datetime.now(timezone.utc).isoformat()+' '+stage+'\n')
        with redirect_stdout(stream), redirect_stderr(stream):
            try:
                yield path
            except BaseException:
                traceback.print_exc(file=stream)
                raise
            else:
                stream.write('\nTermine sans erreur.\n')


In [ ]:
"""Enquetes, indicateurs et contrats CSV. Aucun taux national extrapole localement."""
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
import hashlib, html, json, shutil
import numpy as np
import pandas as pd

RESEARCH_SOURCES=[
    dict(source_id='ENA2025',title='ADEME - Enquete nationale autopartage edition 2025',
         url='https://librairie.ademe.fr/mobilite-et-transports/9344-enquete-nationale-autopartage-2025.html',
         publication='2026-05-07',scope='Plusieurs formes et operateurs, pas uniquement Citiz',
         evidence='Notice du rapport ; rapport integral non exploite'),
    dict(source_id='ADEME2025_SYNTHESE',title='ADEME Infos - enseignements ENA2025',
         url='https://infos.ademe.fr/mobilite-transports/2026/autopartage-jusqua-1-000-e-deconomies-par-an/',
         publication='2026-05',scope='Autopartage, dont autopartage en boucle',evidence='Synthese publique'),
    dict(source_id='ENA2022',title='ADEME - resultats de l enquete autopartage 2022',
         url='https://www.ademe.fr/presse/communique-national/enquete-nationale-autopartage-2022-impact-sur-les-pratiques-de-mobilite-des-francais/',
         publication='2022-09-22',scope='2768 reponses ; plusieurs operateurs, dont 12 reseaux Citiz',evidence='Communique de resultats'),
    dict(source_id='CITIZ_ORLEANS2025',title='Citiz - consultation Orleans et deploiement Centre-Val de Loire',
         url='https://www.citiz.coop/uploads/2025/11/citiz-RNTP-2025.pdf',publication='2025-10-20',
         scope='154 repondants volontaires locaux ; intentions declarees',evidence='Communique Citiz, page 2'),
    dict(source_id='CITIZ_USAGES',title='Citiz - livre blanc, usages multiples',
         url='https://www.citiz.coop/uploads/2026/01/Le-livre-blanc-de-l-autopartage-Citiz-12-2025.pdf',
         publication='2025-12',scope='Presentation du service ; pas classement statistique des motivations',
         evidence='Livre blanc Citiz, page 6')]

# Releve manuel controle dans les pages originales. Les valeurs publiees restent
# distinctes des KPI internes : aucun denominateur reconstitue depuis un pourcentage.
RESEARCH_SOURCES += [
    dict(source_id='ENA2022_RAPPORT',title='ADEME / 6t - rapport integral ENA2022',
         url='https://librairie.ademe.fr/ged/7353/enquete-autopartage-2022-rapport.pdf',
         publication='2022-09',scope='Boucle multioperateurs, dont Citiz ; enquete en ligne',
         evidence='Rapport integral, pages 11, 39, 43, 46, 48-49'),
    dict(source_id='FLEURIEU2024',title='Fleurieu-sur-Saone - bulletin municipal numero 49',
         url='https://www.mairie-fleurieusursaone.fr/wp-content/uploads/2024/12/Fleurieu-60p-n%C2%B049.pdf',
         publication='2024-12',scope='Une station Citiz a Fleurieu-sur-Saone, novembre 2023 a octobre 2024',
         evidence='Page PDF 12, tableau Citiz et texte municipal'),
    dict(source_id='ROVALTAIN2024',title='Valence Romans Agglo - rapport environnemental Rovaltain 2024',
         url='https://www.greendrome.fr/data/fm/Rapport_environnemental_rovaltain_2024_final_1.pdf',
         publication='2024',scope='Parc Rovaltain, pas ensemble des communes environnantes',
         evidence='Page PDF 10 ; periode precise de comptage non explicitee'),
    dict(source_id='CITIZ_LB2026',title='Citiz - livre blanc 2026',
         url='https://www.citiz.coop/uploads/2026/02/2026-CITIZ-livre_blanc.pdf',
         publication='2026-02',scope='Satisfaction reseau Citiz 2024 ; effectif non publie ici',
         evidence='Page PDF 13, page imprimee 12, note 14')]


def public_usage_observations():
    """Chiffres publics historiques : references contextuelles, jamais coefficients locaux."""
    rows=[]
    def add(metric,label,value,unit,source_id,page,kind,scope,period,definition,limitation,
            sample=np.nan,numerator=np.nan,denominator=np.nan,approx=False,**geo):
        rows.append(dict(observation_id=source_id+'__'+metric,metric=metric,indicateur=label,value=value,
                         unit=unit,source_id=source_id,page_pdf=page,nature=kind,territoire=scope,
                         periode=period,definition=definition,limitation=limitation,sample_n=sample,
                         numerator=numerator,denominator=denominator,approximate=approx,
                         commune_code=geo.get('commune_code',''),department=geo.get('department',''),
                         region=geo.get('region',''),scope_type=geo.get('scope_type','national'),
                         period_start=geo.get('period_start',''),period_end=geo.get('period_end',''),
                         period_precision=geo.get('period_precision','libelle_source'),
                         strict_operational_kpi=False,used_in_ranking=False))
    national=dict(period_start='2022-05-11',period_end='2022-06-13',period_precision='collecte_jour')
    for metric,label,pct,count in [
        ('motivation_entretien','Eviter les contraintes d entretien',83.,1504),
        ('motivation_cout','Moins cher qu une voiture personnelle',80.,1451),
        ('motivation_ecologie','Dimension ecologique',77.4,1403),
        ('motivation_stationnement','Eviter les problemes de stationnement',65.6,1189),
        ('motivation_pratique_tc','Plus pratique que les transports en commun',44.6,808)]:
        add(metric,label,pct,'%', 'ENA2022_RAPPORT',46,'enquete_declarative',
            'France - inscrits boucle, plusieurs operateurs','Collecte mai-juin 2022',
            'Element cite dans le choix d inscription ; plusieurs reponses possibles.',
            '1813 inscrits, dont 70 non-utilisateurs sur 12 mois ; non representatif des habitants de chaque commune.',
            sample=1813,numerator=count,denominator=1813,**national)
    for metric,label,pct,count in [
        ('declencheur_tc','Inscription en complement des TC',19.7,358),
        ('declencheur_remplacement','Besoin de remplacer une voiture',17.5,317),
        ('declencheur_velo','Inscription en complement du velo',13.6,246)]:
        add(metric,label,pct,'%','ENA2022_RAPPORT',43,'enquete_declarative',
            'France - inscrits boucle, plusieurs operateurs','Collecte mai-juin 2022',
            'Evenement declencheur de l inscription, figure 35 ; distinct des avantages cites figure 37.',
            'Enquete retrospective ; taux nationaux non appliques aux communes.',
            sample=1813,numerator=count,denominator=1813,**national)
    add('sans_voiture','Aucune voiture disponible dans le menage',73.8,'%','ENA2022_RAPPORT',39,
        'enquete_declarative','France - inscrits boucle, plusieurs operateurs','Collecte mai-juin 2022',
        'Aucune voiture a disposition dans le menage, voiture de fonction comprise, figure 29.',
        'Inscrits interroges ; ne mesure pas le nombre local de personnes ayant le permis sans vehicule.',
        sample=1813,numerator=1338,denominator=1813,**national)
    for metric,label,value in [('locations_mois_moyenne','Utilisations mensuelles, moyenne',2.8),
                               ('locations_mois_mediane','Utilisations mensuelles, mediane',2.)]:
        add(metric,label,value,'utilisations/personne/mois','ENA2022_RAPPORT',49,
            'frequence_estimee_depuis_declarations','France - utilisateurs actifs boucle','Collecte mai-juin 2022',
            'Conversion des classes de frequence declaree en utilisations mensuelles, figures 43-44.',
            '1743 utilisateurs actifs : au moins une utilisation sur 12 mois ; pas un journal de reservations.',
            sample=1743,**national)
    local=dict(commune_code='69085',department='69',region='84',scope_type='commune',
               period_start='2023-11-01',period_end='2024-10-31',period_precision='mois_source_bornes_calendaires')
    for metric,label,value,unit,definition in [
        ('locations','Locations publiees',94,'locations','Nombre de locations sur les douze mois.'),
        ('kilometres','Kilometres parcourus',8274,'km','Kilometres des deux vehicules sur la periode.'),
        ('heures_reservees','Heures reservees',999,'heures','Duree reservee publiee ; pas temps de conduite.'),
        ('vehicules','Vehicules presentes dans le bilan',2,'vehicules','Flotte presentee dans le tableau ; jours reels de service inconnus.'),
        ('stations','Station concernee',1,'station','Station Fleurieu-sur-Saone presentee dans le bilan.'),
        ('clients_actifs','Clients actifs publies',15,'clients','Le bilan indique 15 actifs ; seuil d activite non explicite.'),
        ('locations_jour_vehicule_publie','Taux de locations publie',.13,'locations/jour/vehicule',
         'Taux fourni dans le document ; denominateur et indisponibilites non documentes.'),
        ('trajet_moyen_km','Distance moyenne publiee',88,'km/location','Moyenne arrondie publiee ; pas recalculee.'),
        ('trajet_moyen_heures','Duree moyenne publiee',11,'heures/location','Environ 11 heures ; valeur approximative publiee.')]:
        add(metric,label,value,unit,'FLEURIEU2024',12,'bilan_exploitation_publie','Fleurieu-sur-Saone',
            'Novembre 2023 - octobre 2024',definition,
            'Une station historique ; aucune extrapolation nationale, aucun bonus de priorite. Jours de service non publies.',
            approx=metric in ['trajet_moyen_km','trajet_moyen_heures'],**local)
    for metric,label,value,unit in [('reservations','Reservations Citiz',409,'reservations'),
                                   ('kilometres','Kilometres Citiz',54906,'km')]:
        add(metric,label,value,unit,'ROVALTAIN2024',10,'bilan_exploitation_publie','Parc Rovaltain',
            'Bilan 2024 - fenetre exacte non explicitee','Indicateur Citiz du parc dans le rapport environnemental 2024.',
            'Perimetre de site ; ni total communal ni serie annuelle certifiee. Flotte et jours de service non publies.',
            department='26',region='84',scope_type='site')
    for metric,label,value,approx in [('usage_anterieur','Autopartage deja utilise',13,False),
                                      ('achat_evite_intention','Vehicule cede ou achat evite envisage',43,False),
                                      ('usage_intention','Intention d utiliser le service',92,False),
                                      ('besoin_hebdomadaire','Besoin hebdomadaire, pres de 20 %',20,True)]:
        add(metric,label,value,'%','CITIZ_ORLEANS2025',2,'consultation_intention',
            'Metropole d Orleans','Consultation lancee janvier 2025, publiee octobre 2025',
            'Consultation citoyenne volontaire ; pourcentage publie, pas nombre de reservations.',
            '154 repondants au total ; effectif par question non fourni. Aucun effectif deduit des pourcentages arrondis.',
            sample=154,approx=approx,department='45',region='24',scope_type='metropole')
    add('satisfaction','Utilisateurs satisfaits',92,'%','CITIZ_LB2026',13,'enquete_declarative',
        'Reseau Citiz','Enquete de satisfaction 2024, reprise dans le livre blanc 2026',
        'Satisfaction publiee avec renvoi a l enquete annuelle Citiz 2024 (note 14, page imprimee 12).',
        'Echantillon, taux de reponse et questionnaire non fournis dans ce document.')
    add('demotorisation_apres_inscription','Vehicule cede apres inscription',60,'%','ADEME2025_SYNTHESE',np.nan,
        'enquete_declarative','France - utilisateurs autopartage en boucle','ENA2025, synthese mai 2026',
        'Part des usagers en boucle declarant s etre separes d un vehicule apres inscription.',
        'Effectif de cette question absent de la synthese ; evolution retrospective non causale.')
    add('notoriete_conducteurs','Conducteurs connaissant l autopartage',59,'%','ADEME2025_SYNTHESE',np.nan,
        'enquete_declarative','France - conducteurs','ENA2025, synthese mai 2026',
        'Connaissance et comprehension de l autopartage dans le volet conducteurs representatif de 1000 personnes.',
        'Pas un taux d intention ni d activation ; nombre de reponses par question non fourni.',sample=1000)
    result=pd.DataFrame(rows)
    refs=pd.DataFrame(RESEARCH_SOURCES)[['source_id','url','publication']].rename(columns={'url':'source_url','publication':'source_publication'})
    result=result.merge(refs,on='source_id',how='left',validate='many_to_one')
    result['verification']='releve_verifie_dans_source_publique'
    result['verified_at']='2026-10-21'
    validate_public_observations(result)
    return result


def validate_public_observations(data):
    if data.observation_id.duplicated().any(): raise ValueError('Observation publique dupliquee.')
    if data.source_url.isna().any() or not data.source_url.str.startswith('https://').all(): raise ValueError('Source publique requise.')
    if (~np.isfinite(data.value)|data.value.lt(0)).any(): raise ValueError('Valeur publique invalide.')
    if (data.unit.eq('%')&data.value.gt(100)).any(): raise ValueError('Pourcentage public invalide.')
    count_pair=data.numerator.notna()&data.denominator.gt(0)
    if (abs(100*data.loc[count_pair,'numerator']/data.loc[count_pair,'denominator']-data.loc[count_pair,'value'])>.11).any():
        raise ValueError('Effectifs incoherents avec la valeur publiee arrondie.')
    if data.used_in_ranking.any() or data.strict_operational_kpi.any(): raise ValueError('Observation contextuelle confondue avec KPI local.')
    if (data.scope_type.ne('commune')&data.commune_code.ne('')).any(): raise ValueError('Rattachement communal abusif.')
    return data


def public_observations_for_scope(data,codes,regions=(),departments=()):
    """References nationales toujours visibles, locales seulement dans leur perimetre."""
    match=data.scope_type.eq('national')
    match|=data.scope_type.eq('commune')&data.commune_code.isin(codes)
    match|=data.scope_type.isin(['site','metropole'])&data.department.isin(departments)
    return data.loc[match].copy()


def research_findings():
    rows=[
        ('economie','Economie et absence de contraintes de possession','ADEME2025_SYNTHESE',np.nan,np.nan,'Motivation qualitative ; pas de coefficient local'),
        ('occasionnel','Achats, visites et sorties ; locations courtes en boucle','ENA2022',np.nan,2768,'Echantillon multioperateurs ; pas 2768 usagers Citiz'),
        ('complement_tc','Voiture occasionnelle en complement des TC et du velo','ADEME2025_SYNTHESE',np.nan,np.nan,'Le manque de TC au domicile ne suffit pas a decrire le besoin'),
        ('achat_evite','Abandon d un vehicule ou achat evite envisage','CITIZ_ORLEANS2025',43.,154,'Intention locale ; non extrapolable a la France'),
        ('hebdomadaire','Besoin hebdomadaire declare, pres de 20 pour cent','CITIZ_ORLEANS2025',np.nan,154,'Valeur approximative conservee dans le texte'),
        ('vehicule_adapte','Utilitaire, groupe ou vehicule adapte selon le besoin','CITIZ_USAGES',np.nan,np.nan,'Usages presentes par l operateur, pas frequence mesuree')]
    return pd.DataFrame(rows,columns=['theme','constat','source_id','pourcentage','effectif_total','limite'])


def metric_catalog():
    rows=[
        ('public_motivations','Motivations declarees en boucle','public_collecte','pourcentage','ADEME / 6t ENA2022 p46','1813 inscrits ; plusieurs operateurs ; choix multiples'),
        ('public_station_activity','Locations, heures et kilometres de bilans Citiz','public_collecte','comptages','Fleurieu2024 / Rovaltain2024','Observations historiques de sites ; pas une couverture nationale'),
        ('public_satisfaction','Satisfaction du reseau Citiz','public_collecte','pourcentage','Citiz livre blanc 2026, enquete2024','92 pour cent ; effectif non publie dans le document'),
        ('target_license_no_vehicle','Cible permis sans vehicule','estimation','personnes','EMP2019 + RP2022','Profil regional applique localement ; non observe au carreau'),
        ('target_gap_share','Cible eloignee des arrets publies','proxy','part','Arrets PAN + grille','Ni frequence ni trajet TC ; inventaire a auditer'),
        ('walking_access','Acces pieton reel a une station','a_collecter','minutes','Reseau pieton + stations','Les distances actuelles sont aeriennes'),
        ('tc_useful_trips','Trajets utiles difficiles en TC','a_collecter','part','GTFS horaires + destinations','Soirees, dimanche et motifs a distinguer'),
        ('usage_basket_cost','Cout de paniers d usage','a_collecter','EUR','Tarifs Citiz sources + duree + km','Pas un budget d implantation ; aucun montant suppose'),
        ('activation_30d','Premiere reservation sous 30 jours','a_collecter','pourcentage','Agregats Citiz','Inscrits ayant tous 30 jours de recul'),
        ('reuse_90d','Nouvelle reservation sous 90 jours','a_collecter','pourcentage','Agregats Citiz','Cohorte de premiers utilisateurs ayant 90 jours de recul'),
        ('unmet_search_rate','Recherches sans vehicule compatible','a_collecter','pourcentage','Recherches Citiz dedoublonnees','Pas deduit d un instantane GBFS'),
        ('bookings_per_vehicle_day','Reservations par vehicule-jour de service','a_collecter','reservations/vehicule-jour','Agregats Citiz','Jours de service reels ; pas nombre de stations'),
        ('mobility_enabled','Deplacement rendu possible','a_collecter','pourcentage','Enquete locale','Declaration de trajet auquel on aurait renonce'),
        ('demotorization_followup','Baisse de motorisation au suivi','a_collecter','pourcentage','Enquete avant/apres','Evolution declaree, pas preuve de causalite')]
    return pd.DataFrame(rows,columns=['metric','indicateur','disponibilite_initiale','unite','donnees_requises','definition'])


KPI_COLUMNS=['commune_code','period_start','period_end','metric','numerator','denominator','source','definition_version']
SURVEY_COLUMNS=['commune_code','motif','n_reponses','n_eligibles','date_debut','date_fin','source']
MOTIFS=['courses_volumineuses','visites_proches','sorties_loisirs','trajet_tardif','destination_mal_desservie',
        'accompagnement','professionnel','vehicule_adapte','achat_evite','deplacement_rendu_possible']
KPI_IDS=['activation_30d','reuse_90d','unmet_search_rate','bookings_per_vehicle_day','mobility_enabled','demotorization_followup']


def annotate_evidence(frame):
    data=frame.copy()
    valid=data.priority_score.notna()
    data['evidence_status']=np.where(valid,'limitee_modele_non_valide','insuffisante_score_absent')
    data['evidence_gaps']='horaires_TC_non_mesures;cheminement_pieton_non_calcule;usage_occasionnel_non_mesure;parking_non_verifie'
    if 'local_survey_status' not in data: data['local_survey_status']='a_collecter'
    data['feasibility_status']='parking_et_implantation_a_verifier'
    if 'cells' in data and 'imputed_cells' in data:
        data['imputed_cell_share']=(data.imputed_cells/data.cells.where(data.cells>0)).clip(0,1)
    else: data['imputed_cell_share']=np.nan
    data['license_profile_age_years']=datetime.now(timezone.utc).year-2019
    return data


def _validate_aggregate(data,columns,keys,start,end):
    missing=set(columns)-set(data.columns)
    if missing: raise ValueError('Colonnes manquantes : '+', '.join(sorted(missing)))
    if set(data.columns)-set(columns): raise ValueError('Colonnes inattendues : importer uniquement le contrat agrege.')
    data=data.copy()
    data['commune_code']=data.commune_code.astype('string')
    if not data.commune_code.str.fullmatch(r'[0-9AB]{5}',na=False).all(): raise ValueError('Code commune INSEE a 5 caracteres requis.')
    if data.source.fillna('').astype(str).str.strip().eq('').any(): raise ValueError('Source requise pour chaque agregat.')
    dates={c:pd.to_datetime(data[c],errors='coerce',utc=True) for c in [start,end]}
    if any(s.isna().any() for s in dates.values()) or (dates[start]>dates[end]).any(): raise ValueError('Dates invalides ou inversees.')
    if data.duplicated(keys).any(): raise ValueError('Agregats dupliques pour un meme perimetre et une meme periode.')
    return data


def operational_kpis(data=None):
    if data is None or data.empty:
        return pd.DataFrame([dict(commune_code='',period_start='',period_end='',metric=k,numerator=np.nan,
                                 denominator=np.nan,source='',definition_version='v1',value=np.nan,status='a_collecter') for k in KPI_IDS])
    data=_validate_aggregate(data,KPI_COLUMNS,['commune_code','period_start','period_end','metric'],'period_start','period_end')
    if not data.metric.isin(KPI_IDS).all() or not data.definition_version.eq('v1').all(): raise ValueError('Metrique ou version de definition inconnue.')
    for c in ['numerator','denominator']:
        data[c]=pd.to_numeric(data[c],errors='raise')
        if (~np.isfinite(data[c])|data[c].lt(0)).any(): raise ValueError('Effectifs finis positifs ou nuls requis.')
    rates=data.metric.ne('bookings_per_vehicle_day')
    if (rates & data.numerator.gt(data.denominator)).any(): raise ValueError('Numerateur superieur au denominateur pour un taux.')
    if data.loc[rates,['numerator','denominator']].mod(1).ne(0).any().any(): raise ValueError('Effectifs entiers requis pour les taux.')
    if (data.denominator.eq(0)&data.numerator.gt(0)).any(): raise ValueError('Numerateur positif sans exposition.')
    data['value']=data.numerator/data.denominator.where(data.denominator>0)*np.where(rates,100,1)
    data['status']=np.where(data.denominator.gt(0),'calcule_depuis_agregats_sources','denominateur_nul_non_calculable')
    return data


def survey_metrics(data=None):
    if data is None or data.empty: return pd.DataFrame(columns=SURVEY_COLUMNS+['share_pct','status'])
    data=_validate_aggregate(data,SURVEY_COLUMNS,['commune_code','motif','date_debut','date_fin'],'date_debut','date_fin')
    if not data.motif.isin(MOTIFS).all(): raise ValueError('Motif inconnu.')
    for c in ['n_reponses','n_eligibles']:
        data[c]=pd.to_numeric(data[c],errors='raise')
        if (~np.isfinite(data[c])|data[c].lt(0)|data[c].mod(1).ne(0)).any(): raise ValueError('Effectifs d enquete entiers positifs ou nuls requis.')
    if data.n_reponses.gt(data.n_eligibles).any(): raise ValueError('Reponses superieures aux repondants eligibles.')
    data['share_pct']=100*data.n_reponses/data.n_eligibles.where(data.n_eligibles>0)
    data['status']=np.where(data.n_eligibles.gt(0),'declaratif_echantillon_local_non_representativite_garantie','denominateur_nul_non_calculable')
    return data


def archive_research_sources(workdir):
    from concurrent.futures import ThreadPoolExecutor
    raw=Path(workdir)/'research_raw'; raw.mkdir(parents=True,exist_ok=True)
    def collect(source):
        ext='.pdf' if source['url'].endswith('.pdf') else '.html'
        file=raw/(source['source_id']+ext)
        record={**source,'file':file.name}
        try:
            if not file.exists():
                with urlopen(Request(source['url'],headers={'User-Agent':'CitizResearchPrototype/1.0'}),timeout=25) as response:
                    content=response.read()
                if ext=='.pdf' and not content.startswith(b'%PDF'): raise ValueError('Reponse non PDF')
                file.write_bytes(content)
            record.update(status='archive_publique',sha256=hashlib.sha256(file.read_bytes()).hexdigest(),
                          retrieved_at=datetime.fromtimestamp(file.stat().st_mtime,timezone.utc).isoformat())
        except Exception as error: record.update(status='indisponible',detail=str(error))
        return record
    with ThreadPoolExecutor(max_workers=4) as pool: records=list(pool.map(collect,RESEARCH_SOURCES))
    meta=Path(workdir)/'bronze_metadata'; meta.mkdir(exist_ok=True)
    pd.DataFrame(records).to_csv(meta/'enquetes_sources_manifest.csv',index=False,encoding='utf-8')
    return pd.DataFrame(records)


def prepare_mobility_metrics(france):
    root=Path(france['workdir']); prepared=root/'silver_metadata'; prepared.mkdir(exist_ok=True)
    output=root/'output'; output.mkdir(exist_ok=True)
    templates=root/'templates'; templates.mkdir(exist_ok=True)
    research_findings().to_csv(prepared/'enquetes_constats.csv',index=False,encoding='utf-8')
    pd.DataFrame(RESEARCH_SOURCES).to_csv(prepared/'enquetes_references.csv',index=False,encoding='utf-8')
    observations=public_usage_observations()
    manifest=root/'bronze_metadata'/'enquetes_sources_manifest.csv'
    if manifest.exists():
        provenance=pd.read_csv(manifest).reindex(columns=['source_id','status','sha256','retrieved_at'])
        observations=observations.merge(provenance.rename(columns={'status':'archive_status'}),on='source_id',how='left',validate='many_to_one')
    else:
        observations=observations.assign(archive_status='archivage_a_executer',sha256='',retrieved_at='')
    observations.to_csv(prepared/'usages_observations_publiques.csv',index=False,encoding='utf-8')
    observations.to_csv(output/'citiz_fr_usages_publics.csv',index=False,encoding='utf-8')
    metric_catalog().to_csv(output/'citiz_fr_metriques_catalogue.csv',index=False,encoding='utf-8')
    frames={}; inputs={}
    for filename,columns,calculate in [('kpi_agreges.csv',KPI_COLUMNS,operational_kpis),('enquete_agregee.csv',SURVEY_COLUMNS,survey_metrics)]:
        template=templates/filename
        if not template.exists(): pd.DataFrame(columns=columns).to_csv(template,index=False,encoding='utf-8')
        path=root/'inputs'/filename
        data=pd.read_csv(path,dtype={'commune_code':str}) if path.exists() else None
        result=calculate(data)
        inputs[filename]=(path,data)
        frames[filename]=result
    known=set(france['overview'].commune_code)
    for frame in frames.values():
        codes=set(frame.commune_code.dropna())-{''}
        if codes-known: raise ValueError('Agregat avec code commune absent du perimetre : '+', '.join(sorted(codes-known)))
    for filename,(path,data) in inputs.items():
        if data is not None and not data.empty:
            raw=root/'research_inputs'; raw.mkdir(exist_ok=True); shutil.copyfile(path,raw/filename)
            data.to_csv(prepared/filename,index=False,encoding='utf-8')
    local_codes=set(frames['enquete_agregee.csv'].loc[frames['enquete_agregee.csv'].n_eligibles.gt(0),'commune_code'])
    france['overview']['local_survey_status']=np.where(france['overview'].commune_code.isin(local_codes),
                                                      'enquete_locale_declarative','a_collecter')
    frames['kpi_agreges.csv'].to_csv(output/'citiz_fr_kpi_suivi.csv',index=False,encoding='utf-8')
    frames['enquete_agregee.csv'].to_csv(output/'citiz_fr_enquete_locale.csv',index=False,encoding='utf-8')
    questions=[
        ('age_band','Dans quelle tranche d age vous situez-vous ?','18_24;25_34;35_44;45_59;60_plus;non_reponse'),
        ('usage_actuel','Utilisez-vous actuellement un service d autopartage ?','oui;non;ancien_usager;non_reponse'),
        ('permis_B','Possedez-vous un permis B utilisable pour le vehicule propose ?','oui;non;non_reponse'),
        ('vehicule_disponible','Disposez-vous personnellement d un vehicule lorsque vous en avez besoin ?','toujours;parfois;jamais;non_reponse'),
        ('frequence','Combien de besoins occasionnels de voiture par mois ?','0;1;2_3;4_plus;non_reponse'),
        ('motifs','Pour quels motifs utiliseriez-vous une voiture partagee ? Plusieurs choix possibles.',';'.join(MOTIFS)),
        ('creneaux','A quels moments ?','semaine_jour;semaine_soir;samedi;dimanche;non_reponse'),
        ('solution_actuelle','Comment realisez-vous ces deplacements actuellement ?','TC;velo;voiture_empruntee;location;taxi;renoncement;autre;non_reponse'),
        ('marche','Quelle duree de marche accepteriez-vous pour rejoindre une voiture ?','minutes_ou_non_reponse'),
        ('freins','Quels freins a l utilisation ?','prix;distance;disponibilite;connaissance;vehicule_inadapte;autre;non_reponse'),
        ('achat_evite','Le service pourrait-il vous permettre d eviter un achat ou de ceder un vehicule ?','oui;non;incertain;non_reponse')]
    pd.DataFrame(questions,columns=['variable','question','modalites']).to_csv(templates/'questionnaire_local.csv',index=False,encoding='utf-8')
    france['mobility']={'findings':research_findings(),'references':pd.DataFrame(RESEARCH_SOURCES),'catalog':metric_catalog(),
                       'kpis':frames['kpi_agreges.csv'],'survey':frames['enquete_agregee.csv'],'public_observations':observations}
    return france['mobility']


def mobility_panel_html(france,ranked):
    mobility=prepare_mobility_metrics(france)
    esc=html.escape
    sources=''.join(f'<li><a href="{esc(r.url,quote=True)}" target="_blank" rel="noopener">{esc(r.title)}</a> — {esc(r.scope)}. {esc(r.evidence)}.</li>' for r in mobility['references'].itertuples())
    codes=set(ranked.commune_code) if 'commune_code' in ranked else set()
    kpis=mobility['kpis']; kpis=kpis.loc[kpis.commune_code.isin(codes)|kpis.commune_code.eq('')]
    survey=mobility['survey']; survey=survey.loc[survey.commune_code.isin(codes)]
    observations=public_observations_for_scope(mobility['public_observations'],codes,
        departments=set(ranked.department) if 'department' in ranked else set())
    def show_observations(frame):
        if frame.empty: return '<p>Aucune observation publique collectee dans ce perimetre.</p>'
        body=[]
        for r in frame.itertuples():
            value=('environ ' if r.approximate else '')+f'{r.value:g} '+esc(r.unit)
            sample=f'Echantillon total : {r.sample_n:g}' if pd.notna(r.sample_n) else 'Effectif non fourni'
            page=f' · PDF p. {int(r.page_pdf)}' if pd.notna(r.page_pdf) else ''
            context=esc(r.territoire)+'<br><small>'+esc(r.periode)+'<br>'+sample+'</small>'
            source=f'<a href="{esc(r.source_url,quote=True)}" target="_blank" rel="noopener">{esc(r.source_id)}</a>'+page
            body.append('<tr>'+''.join('<td style="padding:10px;border-bottom:1px solid #dce5e5;vertical-align:top;overflow-wrap:anywhere">'+s+'</td>'
                                      for s in [esc(r.indicateur),'<b>'+value+'</b>',context,source])+'</tr>')
        return ('<table style="width:100%;table-layout:fixed;border-collapse:collapse;font:12px/1.5 Segoe UI">'
                '<colgroup><col style="width:27%"><col style="width:16%"><col style="width:37%"><col style="width:20%"></colgroup>'
                '<thead style="background:#edf3f4;text-align:left"><tr><th>Indicateur</th><th>Valeur publiee</th>'
                '<th>Perimetre et periode</th><th>Source</th></tr></thead><tbody>'+''.join(body)+'</tbody></table>')
    public_panel=(f'<h3>Collecte publique : {len(mobility["public_observations"])} chiffres verifies, '
                  f'{mobility["public_observations"].source_id.nunique()} documents</h3>'
                  '<p>References nationales et historiques. Plusieurs operateurs dans ENA2022 ; seuls les bilans Citiz sont specifiques a Citiz. '
                  'Les pourcentages ne sont ni des predictions locales ni des poids de classement.</p>'
                  '<div style="padding:14px;background:#edf7f4;border-left:5px solid #087f70">'
                  '<b>Motivations en boucle, ENA2022 :</b> entretien evite 83 % · cout 80 % · stationnement 65,6 %. '
                  '1813 inscrits interroges ; plusieurs reponses possibles.</div>'
                  '<h4>Enquetes et intentions publiees</h4>'+show_observations(observations.loc[observations.nature.ne('bilan_exploitation_publie')])+
                  '<h4>Utilisation Citiz observee dans les bilans publics</h4>'+show_observations(observations.loc[observations.nature.eq('bilan_exploitation_publie')])+
                  '<p>Les bilans de sites du departement sont des references contextuelles, pas un total pour les communes filtrees. '
                  'Le taux 0,13 de Fleurieu est publie sans jours reels de service ; il ne remplit pas le KPI strict reservations/vehicule-jour. '
                  'Rovaltain : bilan 2024, fenetre exacte non explicitee. Effectifs par question inconnus conserves vides. '
                  'Definitions et limites par ligne dans citiz_fr_usages_publics.csv (Or).</p>')
    return ("<div style='width:100%;max-width:900px;overflow-x:auto;white-space:normal'><h2>Usages, enquetes et suivi</h2><p><b>Besoin occasionnel : achats, visites, sorties, accompagnement et vehicule adapte.</b> "
            "Le manque de TC au domicile est un signal partiel. La priorite TC actuelle reste un scenario de pre-evaluation, pas une prediction de reservations. "
            "Les proportions nationales et la consultation d Orleans ne sont pas appliquees aux communes.</p>"+public_panel+
            "<h3>Pourquoi utiliser l autopartage ?</h3>"+mobility['findings'].to_html(index=False,escape=True,na_rep='Non chiffre')+
            '<h3>Sources et perimetres</h3><ul>'+sources+'</ul><h3>Indicateurs et donnees necessaires</h3>'+
            mobility['catalog'].to_html(index=False,escape=True)+
            '<h3>Suivi operationnel du perimetre</h3><p>Les lignes ci-dessous restent a collecter sans agregats sources. Fenetres 30/90 jours : choix de protocole, pas resultat d enquete.</p>'+
            kpis.round(2).to_html(index=False,escape=True,na_rep='A collecter')+
            '<h3>Enquete locale</h3>'+('<p>A collecter : aucun sondage local importe pour ce perimetre.</p>' if survey.empty else survey.round(2).to_html(index=False,escape=True,na_rep='Non calculable'))+
            '<p>Modeles CSV dans Files/citiz_france/templates (Or). Deposer uniquement les agregats dans Files/citiz_france/inputs (Bronze), puis relancer le notebook. '
            'Questionnaire prepare ; aucune enquete envoyee. Multi-choix : les parts par motif peuvent depasser 100 % au total. Aucun changement automatique des poids.</p></div>')


In [ ]:
"""Publication CSV OneLake, aucun nouveau fichier Parquet ni table Delta.

Les tables Delta de l'ancien prototype Lyon ne sont pas la sortie nationale.
Un rapport Power BI doit importer ces CSV ou utiliser un Dataflow configure.
"""
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import json

FR_WORKSPACE='0532a204-3933-4804-9064-1224447ebbb3'
FR_LAKEHOUSES={'bronze':'3d7404d6-b36f-4b7c-97bd-aa9627ede34e',
               'silver':'2ff2289f-ff48-4d67-b329-22ea66ba3d52',
               'gold':'e22a30e8-658f-4bfd-81a6-0dbb773d1298'}


def restore_france_cache(workdir='/tmp/citiz_france',department=None,verbose=False):
    """Reprise CSV OneLake apres redemarrage ; grilles departementales a la demande."""
    import notebookutils
    root=Path(workdir); count=0
    def remote(layer,relative):
        return f'abfss://{FR_WORKSPACE}@onelake.dfs.fabric.microsoft.com/{FR_LAKEHOUSES[layer]}/Files/citiz_france/{relative}'
    jobs=[]
    if department is not None:
        folder=remote('silver','cells/'+str(department))
        if notebookutils.fs.exists(folder):
            jobs=[(entry.path,root/'cells'/str(department)/entry.name) for entry in notebookutils.fs.ls(folder) if entry.name.endswith('.csv')]
    elif not (root/'output'/'citiz_fr_communes.csv').exists():
        files=['citiz_fr_communes.csv','citiz_fr_stations.csv','citiz_fr_sources.csv','citiz_fr_quality.csv']
        if all(notebookutils.fs.exists(remote('gold','output/'+name)) for name in files):
            jobs=[(remote('gold','output/'+name),root/'output'/name) for name in files]
            jobs += [(remote('bronze','raw/'+name),root/'raw'/name) for name in ['emp2019.zip','emp2019_dictionary.xlsx'] if notebookutils.fs.exists(remote('bronze','raw/'+name))]
    for src,destination in jobs:
        if destination.exists(): continue
        destination.parent.mkdir(parents=True,exist_ok=True)
        notebookutils.fs.cp(src,'file:'+str(destination),False); count+=1
    if department is None:
        for name in ['kpi_agreges.csv','enquete_agregee.csv']:
            source=remote('bronze','inputs/'+name)
            if notebookutils.fs.exists(source):
                destination=root/'inputs'/name; destination.parent.mkdir(exist_ok=True,parents=True)
                notebookutils.fs.cp(source,'file:'+str(destination),False); count+=1
    if verbose: print('REPRISE CSV OneLake :',count,'fichiers.',flush=True)
    return count

def publish_france_files(france, only_scenario=None, include_raw=False, include_cells=True,verbose=False):
    import notebookutils
    try:
        from citiz_france import STUDY_DEPARTMENTS as allowed_departments
    except ImportError:
        allowed_departments=globals()['STUDY_DEPARTMENTS']
    root=Path(france['workdir'])
    def target(layer,relative):
        return f'abfss://{FR_WORKSPACE}@onelake.dfs.fabric.microsoft.com/{FR_LAKEHOUSES[layer]}/Files/citiz_france/{relative}'
    if only_scenario is not None:
        groups=[(Path(only_scenario),'gold','scenarios/'+Path(only_scenario).name)]
    else:
        groups=[(root/'output','gold','output'),(root/'code','gold','code'),
                (root/'diagnostics','gold','diagnostics'),
                (root/'bronze_metadata','bronze','metadata'),(root/'silver_metadata','silver','prepared'),
                (root/'research_raw','bronze','raw/enquetes'),(root/'research_inputs','bronze','inputs'),
                (root/'templates','gold','templates')]
        if include_cells: groups.append((root/'cells','silver','cells'))
        if include_raw: groups.append((root/'raw','bronze','raw'))
    jobs=[]
    for folder,layer,prefix in groups:
        if not folder.exists(): continue
        for file in folder.rglob('*'):
            if file.is_file() and file.suffix not in {'.part','.parquet'}:
                if prefix=='cells' and file.relative_to(folder).parts[0] not in allowed_departments: continue
                relative=prefix+'/'+file.relative_to(folder).as_posix()
                jobs.append((file,target(layer,relative)))
    for parent in sorted({t.rsplit('/',1)[0] for _,t in jobs}): notebookutils.fs.mkdirs(parent)
    def copy(item):
        file,t=item
        notebookutils.fs.cp('file:'+str(file),t,False)
        return dict(file=str(file.relative_to(root)),size_bytes=file.stat().st_size,target=t)
    with ThreadPoolExecutor(max_workers=4) as pool: records=list(pool.map(copy,jobs))
    manifest=root/'publication_csv.json'
    manifest.write_text(json.dumps(records,ensure_ascii=False,indent=2),encoding='utf-8')
    notebookutils.fs.cp('file:'+str(manifest),target('gold','publication_csv.json'),False)
    if verbose: print('FRANCE CSV PUBLIEE :',len(records),'fichiers OneLake, aucune nouvelle table Delta.',flush=True)
    return records


In [ ]:
with diagnostic_output("/tmp/citiz_france","reprise"):
    try:
        restore_france_cache()
    except ImportError:
        pass  # hors Fabric


In [ ]:
"""France: ingestion par morceaux, pre-evaluation communale, sans Spark.

Une commune est un territoire a etudier, jamais une station ni un gain de
couverture. Les grilles detaillees restent partitionnees par departement.
"""
from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor
from urllib.request import urlopen, Request
from datetime import datetime, timezone
import json, zipfile, re, hashlib, html, shutil
import tempfile
import unicodedata
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from pyproj import Transformer
try:
    from mobility_metrics import annotate_evidence, prepare_mobility_metrics
except ImportError:
    pass  # definitions chargees directement dans le notebook portable

FR_SOURCES = {
    'communes.json': 'https://geo.api.gouv.fr/communes?fields=nom,code,codeDepartement,codeRegion,population,centre,surface&format=json',
    'departements.json': 'https://geo.api.gouv.fr/departements',
    'regions.json': 'https://geo.api.gouv.fr/regions',
    'systems.csv': 'https://raw.githubusercontent.com/MobilityData/gbfs/master/systems.csv',
    'transport_catalog.json': 'https://transport.data.gouv.fr/api/datasets',
    'stops_catalog.json': 'https://www.data.gouv.fr/api/1/datasets/arrets-de-transport-en-france/',
    'population.zip': 'https://www.insee.fr/fr/statistiques/fichier/8735162/Filosofi2021_carreaux_200m_csv.zip',
    'logement.zip': 'https://www.insee.fr/fr/statistiques/fichier/8647012/base-ic-logement-2022_csv.zip',
    'emp2019.zip': 'https://www.statistiques.developpement-durable.gouv.fr/media/5052/download?inline=',
    'emp2019_dictionary.xlsx': 'https://www.statistiques.developpement-durable.gouv.fr/media/5051/download?inline=',
}

MUNICIPAL_PARENTS={**{str(69380+i):'69123' for i in range(1,10)},
                   **{str(75100+i):'75056' for i in range(1,21)},
                   **{str(13200+i):'13055' for i in range(1,17)}}

METROPOLITAN_REGIONS=frozenset(['11','24','27','28','32','44','52','53','75','76','84','93','94'])
METROPOLITAN_DEPARTMENTS=frozenset([f'{i:02d}' for i in range(1,96) if i!=20]+['2A','2B'])
STUDY_REGIONS=METROPOLITAN_REGIONS-{'94'}
STUDY_DEPARTMENTS=METROPOLITAN_DEPARTMENTS-{'2A','2B'}


def study_scope(result):
    """Perimetre actif demande : France metropolitaine hors Corse."""
    data=result['overview']
    result['overview']=data.loc[data.region.astype(str).isin(STUDY_REGIONS)].copy()
    # Flux de reseaux Citiz francais : retrait des coordonnees ultramarines.
    # Cette emprise ne constitue pas un raccordement administratif des stations.
    stations=result['stations']
    result['stations']=stations.loc[stations.latitude.between(42.2,51.3)&stations.longitude.between(-5.3,8.3)].copy()
    if 'license_profiles' in result:
        result['license_profiles']=result['license_profiles'].loc[result['license_profiles'].region_key.ne('corse')].copy()
    result['scope']=dict(name='France metropolitaine hors Corse',regions=sorted(STUDY_REGIONS),
                         stations_filter='reseaux_Citiz_francais_emprise_continentale',
                         communes=len(result['overview']),cells=int(result['overview'].cells.sum()) if 'cells' in result['overview'] else None)
    return result

@dataclass
class FranceScenario:
    budget_eur: float | None = None
    site_cost_eur: float | None = None
    budget_source: str = ''
    cost_source: str = ''
    horizon_years: int | None = None
    max_sites: int = 20
    no_car_weight: float = 35
    transit_weight: float = 35
    population_weight: float = 20
    equity_weight: float = 5
    age_bonus: float = 5
    cost_penalty_weight: float = 25
    transit_coverage_penalty: float = 35
    region: str = ''
    department: str = ''
    search: str = ''
    def validate(self):
        values = [self.no_car_weight,
                  self.transit_weight, self.population_weight, self.equity_weight,
                  self.age_bonus, self.cost_penalty_weight, self.transit_coverage_penalty]
        if not all(np.isfinite(x) and x >= 0 for x in values):
            raise ValueError('Parametres negatifs ou non finis.')
        for value in [self.budget_eur,self.site_cost_eur]:
            if value is not None and (not np.isfinite(value) or value < 0):
                raise ValueError('Montant negatif ou non fini.')
        if self.horizon_years is not None and (self.horizon_years<1 or int(self.horizon_years)!=self.horizon_years):
            raise ValueError('Horizon en annees entieres strictement positif.')
        if not 0 <= self.max_sites <= 1000:
            raise ValueError('Maximum de sites entre 0 et 1000 requis.')
        if not 0 <= self.age_bonus <= 5:
            raise ValueError("Bonus d'age limite a 5 points.")
        if self.no_car_weight+self.transit_weight+self.population_weight+self.equity_weight > 95.000001:
            raise ValueError('Les quatre poids principaux doivent totaliser au plus 95 points.')

def fr_download(url, path, refresh=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not refresh and path.exists() and path.stat().st_size:
        return path
    tmp = path.with_suffix(path.suffix+'.part')
    with urlopen(Request(url, headers={'User-Agent':'CitizHelper-France/2.0'}), timeout=120) as response, tmp.open('wb') as output:
        while block := response.read(1024*1024):
            output.write(block)
    tmp.replace(path)
    return path

def fr_json(url):
    with urlopen(Request(url,headers={'User-Agent':'CitizHelper-France/2.0'}),timeout=45) as response:
        return json.load(response)

def sphere_xyz(lat, lon):
    lat,lon=np.radians(np.asarray(lat,dtype=float)),np.radians(np.asarray(lon,dtype=float))
    return np.column_stack([np.cos(lat)*np.cos(lon),np.cos(lat)*np.sin(lon),np.sin(lat)])

def nearest_metres(tree, lat, lon):
    if tree is None:
        return np.full(len(lat),np.nan)
    chord,_=tree.query(sphere_xyz(lat,lon),workers=2)
    return 2*6_371_008.8*np.arcsin(np.minimum(chord/2,1))

def gbfs_feeds(data):
    body=data.get('data',{})
    if 'feeds' in body:
        return body['feeds']
    for language in ['fr','en',*body]:
        if isinstance(body.get(language),dict) and 'feeds' in body[language]:
            return body[language]['feeds']
    raise ValueError('Decouverte GBFS sans feeds.')

def normalized_name(value):
    value=unicodedata.normalize('NFKD',str(value)).encode('ascii','ignore').decode().lower()
    return re.sub('[^a-z0-9]','',value)

def license_profiles(raw):
    """Croisement OBSERVE BPERMIS/JNBVEH. Agrégats d'enquete, pas registre.

    Coefficient: adultes avec permis B sans vehicule leger disponible / menages
    sans vehicule leger disponible, avec les poids individu et menage respectifs.
    Seuils de fiabilite heuristiques explicites, sans bonus de ruralite.
    """
    with zipfile.ZipFile(raw/'emp2019.zip') as z:
        def read(name,sep,columns):
            with z.open(name) as f:
                return pd.read_csv(f,sep=sep,encoding='cp1252',dtype=str,usecols=columns)
        people=read('k_individu_public_V3.csv',';',['IDENT_MEN','IDENT_IND','BPERMIS','pond_indC'])
        ages=read('tcm_ind_kish_public_V3.csv',';',['ident_ind','AGE']).rename(columns={'ident_ind':'IDENT_IND'})
        households=read('q_menage_public_V3.csv',';',['IDENT_MEN','JNBVEH','pond_menC'])
        geography=read('tcm_men_public_V3.csv',',',['ident_men','REG_res']).rename(columns={'ident_men':'IDENT_MEN'})
    people=people.merge(ages,on='IDENT_IND',validate='one_to_one')
    households=households.merge(geography,on='IDENT_MEN',validate='one_to_one')
    people=people.merge(households[['IDENT_MEN','JNBVEH','REG_res']],on='IDENT_MEN',validate='many_to_one')
    people['age']=pd.to_numeric(people.AGE,errors='coerce')
    people['weight']=pd.to_numeric(people.pond_indC,errors='coerce')
    households['weight']=pd.to_numeric(households.pond_menC,errors='coerce')
    for frame in [people,households]:
        frame['no_vehicle']=pd.to_numeric(frame.JNBVEH,errors='coerce').eq(0)
        frame['region_key']=frame.REG_res.map(normalized_name)
    def profile(key):
        p=people if key=='FR_MET' else people.loc[people.region_key.eq(key)]
        h=households if key=='FR_MET' else households.loc[households.region_key.eq(key)]
        p=p.loc[p.no_vehicle & p.age.ge(18) & p.weight.gt(0)]
        h=h.loc[h.no_vehicle & h.weight.gt(0)]
        known=p.BPERMIS.isin(['1','2'])
        effective=float(h.weight.sum()**2/(h.weight.pow(2).sum())) if len(h) else 0.
        missing=float(p.loc[~known,'weight'].sum()/p.weight.sum()) if len(p) else 1.
        holders=float(p.loc[p.BPERMIS.eq('1'),'weight'].sum())
        denominator=float(h.weight.sum())
        reliable=len(h)>=50 and effective>=30 and missing<=.05
        return dict(region_key=key,sample_no_vehicle_households=len(h),sample_adults=len(p),
                    effective_household_n=effective,missing_license_weight_share=missing,
                    weighted_license_holders_no_vehicle=holders,weighted_no_vehicle_households=denominator,
                    licensed_adults_per_no_vehicle_household=holders/denominator if reliable and denominator>0 else np.nan,
                    reliable=reliable,reference_year=2019,scope='France_metropolitaine',
                    definition='BPERMIS=1_AGE>=18_JNBVEH=0_VP_VUL_campingcar',
                    source_url='https://www.statistiques.developpement-durable.gouv.fr/resultats-detailles-de-lenquete-mobilite-des-personnes-de-2019')
    return pd.DataFrame([profile('FR_MET'),*[profile(k) for k in sorted(households.region_key.dropna().unique())]])

def attach_license_profile(result):
    root=Path(result['workdir']); raw=root/'raw'
    for name in ['emp2019.zip','emp2019_dictionary.xlsx']:
        fr_download(FR_SOURCES[name],raw/name)
    profiles=license_profiles(raw)
    national=profiles.loc[profiles.region_key.eq('FR_MET')].iloc[0]
    if not national.reliable: raise ValueError('Echantillon national permis sans vehicule insuffisant.')
    by_region=profiles.set_index('region_key')
    data=result['overview'].copy()
    keys=data.region_name.map(normalized_name)
    coefficients=keys.map(by_region.licensed_adults_per_no_vehicle_household)
    metropolitan=~data.department.astype(str).str.startswith(('97','98')) & data.department.notna()
    use_national=metropolitan & coefficients.isna()
    coefficients.loc[use_national]=national.licensed_adults_per_no_vehicle_household
    coefficients.loc[~metropolitan]=np.nan
    data['license_coefficient']=coefficients
    data['license_status']=np.where(~metropolitan,'hors_champ_EMP2019',np.where(use_national,'estimation_EMP2019_calage_national','estimation_EMP2019_calage_regional'))
    data['license_source']=national.source_url
    data['license_reference_year']=2019
    data['uncovered_licensed_adults_no_car_estimate']=data.uncovered_no_car_proxy*coefficients
    data['uncovered_gap_licensed_adults_estimate']=data.uncovered_gap_no_car_proxy*coefficients
    data['license_local_observed']=False
    result['overview']=data; result['license_profiles']=profiles
    rows=[]
    for name in ['emp2019.zip','emp2019_dictionary.xlsx']:
        rows.append(dict(source=name,url=FR_SOURCES[name],status='ok',retrieved_at=datetime.fromtimestamp((raw/name).stat().st_mtime,timezone.utc).isoformat()))
    result['sources']=pd.concat([result['sources'],pd.DataFrame(rows)],ignore_index=True).drop_duplicates(['source','url'],keep='last')
    return result

def acquire_france(workdir='/tmp/citiz_france', refresh_live=False):
    root=Path(workdir); raw=root/'raw'
    # Reutilisation du cache public V1; aucune dependance a un resultat lyonnais.
    for name in ['population.zip','logement.zip']:
        old=Path('/tmp/citiz_v1/raw')/name
        if old.exists() and not (raw/name).exists():
            raw.mkdir(parents=True,exist_ok=True); shutil.copyfile(old,raw/name)
    def acquire(item):
        name,url=item
        live=refresh_live and name.endswith(('.json','.csv'))
        fr_download(url,raw/name,refresh=live)
        print('France / source disponible:',name,flush=True)
    with ThreadPoolExecutor(max_workers=6) as pool:
        list(pool.map(acquire,FR_SOURCES.items()))
    cat=json.loads((raw/'stops_catalog.json').read_text())
    resource=next(r for r in cat['resources'] if r['format'].lower()=='csv' and r['type']=='main')
    fr_download(resource['url'],raw/'stops.csv',refresh=refresh_live)
    manifest=[dict(source=n,url=u,retrieved_at=datetime.fromtimestamp((raw/n).stat().st_mtime,timezone.utc).isoformat(),status='ok') for n,u in FR_SOURCES.items()]
    manifest.append(dict(source='stops.csv',url=resource['url'],retrieved_at=resource['last_modified'],status='inventaire_experimental_sans_horaires'))
    # Catalogues PAN + MobilityData; exclure le GBFS velo Yelo mal nomme.
    systems=pd.read_csv(raw/'systems.csv').fillna('')
    urls={r['Auto-Discovery URL']:r['Name'] for _,r in systems.iterrows()
          if r['Country Code']=='FR' and 'backend.citiz.fr/public/provider/' in r['Auto-Discovery URL']}
    catalog=json.loads((raw/'transport_catalog.json').read_text())
    for dataset in catalog:
        if 'citiz' in dataset.get('title','').lower() and dataset.get('type')=='vehicles-sharing':
            for r in dataset.get('resources',[]):
                if r.get('format','').lower()=='gbfs':
                    urls[r.get('original_url',r['url'])]=dataset['title']
    # Plusieurs catalogues peuvent pointer le meme provider avec v3.0 / 3.0.
    unique={}
    for url,name in urls.items():
        provider=re.search(r'/provider/(\d+)/',url)
        key='provider_'+provider[1] if provider else url
        unique.setdefault(key,(url,name))
    def station_source(item):
        key,(url,label)=item
        digest=hashlib.sha256(key.encode()).hexdigest()[:12]
        discovery=raw/f'gbfs_{digest}.json'; stations_file=raw/f'stations_{digest}.json'
        try:
            fr_download(url,discovery,refresh_live)
            data=json.loads(discovery.read_text())
            station_url=next(f['url'] for f in gbfs_feeds(data) if f['name']=='station_information')
            fr_download(station_url,stations_file,refresh_live)
            info=json.loads(stations_file.read_text())
            rows=[]
            for station in info['data']['stations']:
                name=station.get('name','')
                if isinstance(name,list): name=' / '.join(n.get('text','') for n in name)
                rows.append(dict(station_id=str(station['station_id']),network=label,provider=key,
                                 name=str(name),latitude=station['lat'],longitude=station['lon']))
            return rows,dict(source=label,url=url,status='ok',retrieved_at=str(info.get('last_updated','')))
        except Exception as error:
            return [],dict(source=label,url=url,status='indisponible_'+type(error).__name__,retrieved_at='')
    station_rows=[]
    with ThreadPoolExecutor(max_workers=6) as pool:
        for rows,status in pool.map(station_source,unique.items()):
            station_rows.extend(rows); manifest.append(status)
            print('GBFS:',status['source'],len(rows),status['status'],flush=True)
    if not station_rows: raise RuntimeError('Aucune station Citiz accessible : couverture inconnue.')
    stations=pd.DataFrame(station_rows)
    # Deduplication geometrique des copies de catalogues, pas du stop_id GTFS.
    stations['lat_key']=stations.latitude.round(5); stations['lon_key']=stations.longitude.round(5)
    stations=stations.drop_duplicates(['lat_key','lon_key']).drop(columns=['lat_key','lon_key'])
    return raw,stations.reset_index(drop=True),pd.DataFrame(manifest)

def load_fr_communes(raw):
    data=json.loads((raw/'communes.json').read_text())
    records=[]
    for r in data:
        coords=r.get('centre',{}).get('coordinates',[np.nan,np.nan])
        records.append(dict(commune_code=r['code'],commune=r['nom'],department=r.get('codeDepartement',''),
                            region=r.get('codeRegion',''),population_legal=r.get('population',np.nan),
                            latitude=coords[1],longitude=coords[0]))
    communes=pd.DataFrame(records).drop_duplicates('commune_code')
    departments={r['code']:r['nom'] for r in json.loads((raw/'departements.json').read_text())}
    regions={r['code']:r['nom'] for r in json.loads((raw/'regions.json').read_text())}
    communes['department_name']=communes.department.map(departments).fillna('Non renseigne')
    communes['region_name']=communes.region.map(regions).fillna('Non renseignee')
    columns=['COM','P22_MEN','P22_RP_VOIT1P','P22_RP_VOIT1']
    with zipfile.ZipFile(raw/'logement.zip') as archive:
        name=next(n for n in archive.namelist() if n.lower().startswith('base-ic') and n.lower().endswith('.csv'))
        with archive.open(name) as f: housing=pd.read_csv(f,sep=';',dtype={'COM':str},usecols=columns)
    for c in columns[1:]: housing[c]=pd.to_numeric(housing[c],errors='coerce')
    # Arrondissements municipaux -> commune mere, avant toute aggregation.
    housing['COM']=housing.COM.replace(MUNICIPAL_PARENTS)
    housing=housing.groupby('COM')[columns[1:]].sum(min_count=1)
    housing['no_car_rate']=(1-housing.P22_RP_VOIT1P/housing.P22_MEN.replace(0,np.nan)).clip(0,1)
    housing['one_car_rate']=(housing.P22_RP_VOIT1/housing.P22_MEN.replace(0,np.nan)).clip(0,1)
    housing['households_rp2022']=housing.P22_MEN
    housing['no_car_households_rp2022']=(housing.P22_MEN-housing.P22_RP_VOIT1P).clip(lower=0)
    communes=communes.merge(housing[['no_car_rate','one_car_rate','households_rp2022','no_car_households_rp2022']],left_on='commune_code',right_index=True,how='left')
    return communes

def prepare_france(workdir, raw, stations, manifest):
    """Preparation 200 m limitee aux communes metropolitaines hors Corse."""
    root=Path(workdir); root.mkdir(parents=True,exist_ok=True)
    silver=Path(tempfile.mkdtemp(prefix='cells_build_',dir=root))
    communes=load_fr_communes(raw)
    communes=communes.loc[communes.region.isin(STUDY_REGIONS)].copy()
    metropolitan_codes=set(communes.commune_code)
    stops_parts=[]
    for chunk in pd.read_csv(raw/'stops.csv',usecols=['stop_lat','stop_lon'],chunksize=250_000):
        for c in chunk: chunk[c]=pd.to_numeric(chunk[c],errors='coerce')
        chunk=chunk.dropna()
        chunk=chunk.loc[chunk.stop_lat.between(-90,90)&chunk.stop_lon.between(-180,180)]
        stops_parts.append(chunk.round(5).drop_duplicates())
    stops=pd.concat(stops_parts,ignore_index=True).drop_duplicates()
    stop_tree=cKDTree(sphere_xyz(stops.stop_lat,stops.stop_lon))
    station_tree=cKDTree(sphere_xyz(stations.latitude,stations.longitude))
    summaries=[]; quality=[]
    usecols=['idcar_200m','lcog_geo','ind','men','men_pauv','ind_18_24','ind_25_39',
             'ind_40_54','ind_55_64','ind_65_79','ind_80p','i_est_200']
    with zipfile.ZipFile(raw/'population.zip') as archive:
        for name in archive.namelist():
            if not name.endswith('.csv'): continue
            with archive.open(name) as stream:
                for chunk_no,chunk in enumerate(pd.read_csv(stream,usecols=usecols,dtype={'lcog_geo':str},chunksize=120_000)):
                    chunk=chunk.loc[(chunk.ind>0)&(chunk.men>0)].copy()
                    if chunk.empty: continue
                    # Aucun partage arbitraire d'un carreau chevauchant des communes.
                    single=chunk.lcog_geo.str.fullmatch(r'[0-9AB]{5}',na=False)
                    quality.append(dict(file=name,chunk=chunk_no,cells=len(chunk),ambiguous_commune=int((~single).sum()),population_ambiguous=float(chunk.loc[~single,'ind'].sum())))
                    chunk=chunk.loc[single].copy()
                    chunk=chunk.rename(columns={'lcog_geo':'commune_code','idcar_200m':'cell_id','ind':'population','men':'households','men_pauv':'poor_households'})
                    chunk['commune_code']=chunk.commune_code.replace(MUNICIPAL_PARENTS)
                    chunk=chunk.loc[chunk.commune_code.isin(metropolitan_codes)].copy()
                    if chunk.empty: continue
                    chunk=chunk.merge(communes[['commune_code','commune','department','region','no_car_rate','one_car_rate']],on='commune_code',how='left',validate='many_to_one')
                    projection=chunk.cell_id.str.extract(r'CRS(\d+)RES200mN(-?\d+)E(-?\d+)')
                    if projection.isna().any().any(): raise ValueError('Projection de grille inconnue.')
                    chunk['latitude']=np.nan; chunk['longitude']=np.nan
                    for crs in projection[0].unique():
                        mask=projection[0].eq(crs)
                        xy=projection.loc[mask,[2,1]].astype(float).to_numpy()+100
                        transform=Transformer.from_crs(int(crs),4326,always_xy=True)
                        lon,lat=transform.transform(xy[:,0],xy[:,1])
                        chunk.loc[mask,'latitude']=lat; chunk.loc[mask,'longitude']=lon
                    chunk['nearest_stop_m']=nearest_metres(stop_tree,chunk.latitude,chunk.longitude)
                    chunk['nearest_citiz_m']=nearest_metres(station_tree,chunk.latitude,chunk.longitude)
                    chunk['citiz_covered']=chunk.nearest_citiz_m.le(400)
                    # Eloignement des arrets PUBLIES, jamais frequence de desserte.
                    chunk['transit_gap_proxy']=(chunk.nearest_stop_m/400-1).clip(0,1)
                    chunk['adults']=chunk[[c for c in usecols if c.startswith('ind_')]].sum(axis=1)
                    chunk['age_25_39']=chunk.ind_25_39
                    chunk['no_car_households_proxy']=chunk.households*chunk.no_car_rate
                    chunk['uncovered_population']=chunk.population*(~chunk.citiz_covered)
                    chunk['uncovered_no_car_proxy']=chunk.no_car_households_proxy*(~chunk.citiz_covered)
                    chunk['uncovered_gap_no_car_proxy']=chunk.uncovered_no_car_proxy*chunk.transit_gap_proxy
                    chunk['population_far_stops']=chunk.population*chunk.nearest_stop_m.gt(400)
                    chunk['license_status']='non_mesure'
                    chunk['transit_status']='proximite_arrets_publies_horaires_non_mesures'
                    sums=['population','households','poor_households','adults','age_25_39','uncovered_population',
                          'uncovered_no_car_proxy','uncovered_gap_no_car_proxy','population_far_stops']
                    summary=chunk.groupby('commune_code')[sums].sum(min_count=1)
                    summary['cells']=chunk.groupby('commune_code').size()
                    summary['imputed_cells']=chunk.groupby('commune_code').i_est_200.sum()
                    summaries.append(summary)
                    for department,group in chunk.groupby('department'):
                        folder=silver/str(department); folder.mkdir(exist_ok=True)
                        destination=folder/'cells.csv'
                        group.drop(columns=[c for c in usecols if c.startswith('ind_')]).to_csv(
                            destination,index=False,encoding='utf-8',mode='a',header=not destination.exists())
                    print('France / grilles:',name,chunk_no,len(chunk),flush=True)
    overview=communes.merge(pd.concat(summaries).groupby(level=0).sum(min_count=1),left_on='commune_code',right_index=True,how='left')
    overview['grid_status']=np.where(overview.cells.notna(),'200m_Filosofi2021','grille_absente_ou_geographie_non_raccordee')
    overview['license_status']='non_mesure'
    overview['transit_status']='proximite_arrets_publies_horaires_non_mesures'
    overview['citiz_inventory_status']='catalogues_partiels_verifier_terrain'
    overview['share_25_39_adults']=(overview.age_25_39/overview.adults.replace(0,np.nan)).clip(0,1)
    overview['poverty_rate']=(overview.poor_households/overview.households.replace(0,np.nan)).clip(0,1)
    # Ne pas confondre RP2022, population legale et Filosofi2021.
    overview['population_reference']='Filosofi2021_pour_score_population_legale_separee'
    # Bascule des partitions seulement apres une preparation complete.
    current=root/'cells'
    if current.exists():
        archive_folder=root/'archive'; archive_folder.mkdir(exist_ok=True)
        current.rename(archive_folder/('cells_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')))
    silver.rename(current)
    return dict(workdir=str(root),overview=overview,stations=stations,stops=stops,
                sources=manifest,quality=pd.DataFrame(quality))

def france_rank(overview, scenario=None, cost_overrides=None):
    scenario=scenario or FranceScenario(); scenario.validate()
    data=overview.copy()
    if scenario.region: data=data.loc[data.region.eq(scenario.region)]
    if scenario.department: data=data.loc[data.department.eq(scenario.department)]
    if scenario.search: data=data.loc[data.commune.str.contains(scenario.search,case=False,regex=False,na=False)|data.commune_code.eq(scenario.search)]
    def signal(column,anchor):
        x=data[column].clip(lower=0)
        return x/(x+anchor)
    target='uncovered_licensed_adults_no_car_estimate' if 'uncovered_licensed_adults_no_car_estimate' in data else 'uncovered_no_car_proxy'
    target_gap='uncovered_gap_licensed_adults_estimate' if 'uncovered_gap_licensed_adults_estimate' in data else 'uncovered_gap_no_car_proxy'
    # Part de la cible restant eloignee des arrets publies. Le volume seul
    # ne suffit pas dans une commune deja bien desservie (ex. Paris).
    gap=(data[target_gap]/data[target].where(data[target]>0)).clip(0,1)
    gap=gap.mask(data[target].eq(0),0)
    data['target_transit_gap']=gap
    support=(data[target]/200).clip(0,1)
    data['points_no_car']=scenario.no_car_weight*signal(target,1000)*gap
    data['points_transit']=scenario.transit_weight*signal(target_gap,500)
    data['target_basis']='permis_sans_vehicule_estime_EMP2019' if target!='uncovered_no_car_proxy' else 'proxy_sans_voiture_permis_non_mesure'
    data['points_population']=scenario.population_weight*signal('uncovered_population',10000)*gap
    data['points_equity']=scenario.equity_weight*(data.poverty_rate/.4).clip(0,1)*gap*support
    data['points_age']=scenario.age_bonus*(data.share_25_39_adults/.4).clip(0,1)*gap*support
    terms=['points_no_car','points_transit','points_population','points_equity','points_age']
    data['benefit_score']=data[terms].sum(axis=1,min_count=len(terms))
    data['transit_coverage_penalty']=scenario.transit_coverage_penalty*(1-gap)
    data['mobility_need_score']=(data.benefit_score-data.transit_coverage_penalty).clip(0,100)
    data['cost_total_eur']=np.nan if scenario.site_cost_eur is None else scenario.site_cost_eur
    data['cost_source']=scenario.cost_source.strip()
    data['budget_source']=scenario.budget_source.strip()
    data['horizon_years']=scenario.horizon_years
    data['cost_status']='non_documente'
    if cost_overrides is not None:
        for code,value in cost_overrides.items():
            source=scenario.cost_source.strip()
            if isinstance(value,tuple): value,source=value
            data.loc[data.commune_code.eq(code),'cost_total_eur']=value
            data.loc[data.commune_code.eq(code),'cost_source']=str(source).strip()
    if data.cost_total_eur.lt(0).any(): raise ValueError('Cout negatif interdit.')
    documented=np.isfinite(data.cost_total_eur)&data.cost_source.ne('')
    finance_ready=(scenario.budget_eur is not None and bool(scenario.budget_source.strip()) and bool(documented.any()) and scenario.horizon_years is not None)
    data.loc[documented,'cost_status']='montant_source_a_verifier'
    # Aucun prix ni budget suppose. Le coefficient de cout est relatif au
    # budget documente, pas a un montant de reference financier invente.
    data['cost_penalty']=np.nan
    data['score_status']='besoin_hors_couts_budget_non_documente'
    data['priority_score']=data.mobility_need_score
    if finance_ready:
        if scenario.budget_eur>0:
            data.loc[documented,'cost_penalty']=scenario.cost_penalty_weight*(data.loc[documented,'cost_total_eur']/scenario.budget_eur).clip(upper=2)
        else: data.loc[documented,'cost_penalty']=0.
        data['priority_score']=(data.mobility_need_score-data.cost_penalty).clip(0,100)
        data['score_status']='besoin_moins_penalite_cout_source'
    data['selection_order']=0
    data['selected']=False
    data=data.sort_values(['priority_score','cost_total_eur','commune_code'],ascending=[False,True,True],na_position='last').reset_index(drop=True)
    spent=0.; chosen=0
    for i,r in data.iterrows():
        if not finance_ready or chosen>=scenario.max_sites: break
        if not np.isfinite(r.cost_total_eur) or not str(r.cost_source).strip(): continue
        if not np.isfinite(r.priority_score) or r.priority_score<=0: continue
        if spent+r.cost_total_eur>scenario.budget_eur: continue
        spent+=r.cost_total_eur; chosen+=1
        data.loc[i,['selected','selection_order']]=[True,chosen]
    data['rank']=np.arange(1,len(data)+1)
    data['decision_level']='commune_a_etudier_pas_emplacement_valide'
    if 'annotate_evidence' in globals(): data=annotate_evidence(data)
    return data

def save_france(result, scenario=None):
    root=Path(result['workdir']); output=root/'output'; output.mkdir(parents=True,exist_ok=True)
    # Separations de responsabilite Bronze / Argent / Or, en CSV.
    bronze_meta=root/'bronze_metadata'; bronze_meta.mkdir(exist_ok=True)
    silver_meta=root/'silver_metadata'; silver_meta.mkdir(exist_ok=True)
    result['sources'].to_csv(bronze_meta/'sources_manifest.csv',index=False,encoding='utf-8')
    silver_columns=['commune_code','commune','department','region','population_legal','latitude','longitude',
                    'department_name','region_name','no_car_rate','one_car_rate','households_rp2022',
                    'no_car_households_rp2022','population','households','poor_households','adults',
                    'age_25_39','uncovered_population','uncovered_no_car_proxy','uncovered_gap_no_car_proxy',
                    'population_far_stops','cells','imputed_cells','grid_status','license_status','transit_status',
                    'citiz_inventory_status','share_25_39_adults','poverty_rate','population_reference']
    silver_columns += [c for c in ['license_coefficient','license_source','license_reference_year','license_local_observed',
                                  'uncovered_licensed_adults_no_car_estimate','uncovered_gap_licensed_adults_estimate'] if c in result['overview']]
    result['overview'][silver_columns].to_csv(silver_meta/'communes_enrichies.csv',index=False,encoding='utf-8')
    result['stations'].to_csv(silver_meta/'stations_nettoyees.csv',index=False,encoding='utf-8')
    result['quality'].to_csv(silver_meta/'qualite_raccordement.csv',index=False,encoding='utf-8')
    if 'license_profiles' in result:
        result['license_profiles'].to_csv(silver_meta/'profils_permis_EMP2019.csv',index=False,encoding='utf-8')
    scenario=scenario or FranceScenario()
    result['ranked']=france_rank(result['overview'],scenario)
    result['tables']={
        'citiz_fr_communes':result['ranked'],
        'citiz_fr_stations':result['stations'],
        'citiz_fr_sources':result['sources'],
        'citiz_fr_quality':result['quality'],
    }
    if 'license_profiles' in result: result['tables']['citiz_fr_permis_profils']=result['license_profiles']
    for name,df in result['tables'].items(): df.to_csv(output/f'{name}.csv',index=False,encoding='utf-8')
    (output/'scenario.json').write_text(json.dumps(asdict(scenario),indent=2),encoding='utf-8')
    if 'scope' in result:
        (output/'scope.json').write_text(json.dumps(result['scope'],ensure_ascii=False,indent=2),encoding='utf-8')
    if 'prepare_mobility_metrics' in globals(): prepare_mobility_metrics(result)
    return result

def run_france(workdir='/tmp/citiz_france', refresh=False, rebuild=False):
    root=Path(workdir)
    cache=root/'output'/'citiz_fr_communes.csv'
    if cache.exists() and not refresh and not rebuild:
        print('France / reprise du cache national prepare.',flush=True)
        result=dict(workdir=str(root),overview=pd.read_csv(cache,dtype={'commune_code':str,'department':str,'region':str}),
                    stations=pd.read_csv(root/'output'/'citiz_fr_stations.csv',dtype={'station_id':str}),
                    sources=pd.read_csv(root/'output'/'citiz_fr_sources.csv').fillna(''),quality=pd.read_csv(root/'output'/'citiz_fr_quality.csv'))
        return study_scope(attach_license_profile(result))
    raw,stations,manifest=acquire_france(workdir,refresh_live=refresh)
    result=save_france(study_scope(attach_license_profile(prepare_france(workdir,raw,stations,manifest))))
    print('FRANCE PRETE:',len(result['overview']),'communes;',len(stations),'stations connues;',int(result['overview'].cells.sum()),'carreaux raccordes.',flush=True)
    return result


In [ ]:
with diagnostic_output("/tmp/citiz_france","initialisation"):
    france=run_france()
    _mobility=prepare_mobility_metrics(france)


In [ ]:
"""Citiz Helper: premier diagnostic reproductible, Python natif, sans Spark.

La population couverte est un calcul geographique, le score est une heuristique.
Les permis ne sont pas observes; montants et references sont requis pour financer.
"""
from __future__ import annotations

from dataclasses import dataclass, asdict
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.parse import urlsplit, urlunsplit
import html
import json
import re
import zipfile

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree


@dataclass
class Config:
    workdir: str = "/tmp/citiz_v1"
    reference_date: str = "2026-10-20"
    epci: str = "200046977"  # Metropole de Lyon, perimetre du guide fourni
    radius_m: float = 400.0  # approximation a vol d'oiseau, PAS une isochrone
    budget_eur: float | None = None
    horizon_years: int = 3
    setup_eur: float | None = None
    vehicle_eur: float | None = None
    annual_operation_eur: float | None = None
    budget_source: str = ''
    cost_source: str = ''
    cost_reference_eur: float = 100_000.0
    max_cost_penalty: float = 25.0
    age_bonus_max: float = 5.0  # classe disponible: 25-39 ans, pas 25-35
    max_stations: int = 10
    target_households_anchor: float = 200.0
    population_anchor: float = 1500.0
    no_car_weight: float = 35.0
    transit_weight: float = 35.0
    population_weight: float = 20.0
    equity_weight: float = 5.0
    transit_coverage_penalty: float = 35.0

    def validate(self):
        amounts=[self.budget_eur,self.setup_eur,self.vehicle_eur,self.annual_operation_eur]
        if any(v is None or not np.isfinite(v) for v in amounts):
            raise ValueError('Renseigner les montants : aucun budget ni cout estime par defaut.')
        if (self.budget_eur>0 or any(v>0 for v in amounts[1:])) and (not self.budget_source.strip() or not self.cost_source.strip()):
            raise ValueError('Reference de budget et source des couts requises.')
        values = [self.radius_m, self.cost_reference_eur,
                  self.target_households_anchor, self.population_anchor]
        if not all(np.isfinite(v) and v > 0 for v in values):
            raise ValueError("Rayon et references doivent etre strictement positifs.")
        if self.budget_eur < 0 or self.horizon_years < 1 or self.max_stations < 0:
            raise ValueError("Budget, horizon ou nombre de stations invalide.")
        if not 0 <= self.age_bonus_max <= 5:
            raise ValueError("Le bonus d'age est borne a 5 points.")
        weights = [self.no_car_weight, self.transit_weight, self.population_weight, self.equity_weight]
        if not all(np.isfinite(w) and w >= 0 for w in weights) or sum(weights) > 95.000001:
            raise ValueError('Poids principaux positifs, somme au plus 95 points.')
        if min(self.setup_eur, self.vehicle_eur, self.annual_operation_eur, self.max_cost_penalty) < 0:
            raise ValueError("Les composantes de cout et leur penalite doivent etre positives.")
        if not np.isfinite(self.transit_coverage_penalty) or self.transit_coverage_penalty<0:
            raise ValueError('Penalite de couverture TC invalide.')


SOURCES = {
    "population.zip": "https://www.insee.fr/fr/statistiques/fichier/8735162/Filosofi2021_carreaux_200m_csv.zip",
    "logement.zip": "https://www.insee.fr/fr/statistiques/fichier/8647012/base-ic-logement-2022_csv.zip",
    "citiz.json": "https://data.grandlyon.com/files/rdata/lpa_mobilite.donnees/gbfs.json",
    "gtfs_catalogue": "https://transport.data.gouv.fr/datasets/horaires-theoriques-du-reseau-transports-en-commun-lyonnais",
}


def public_url(url):
    p = urlsplit(url)
    return urlunsplit((p.scheme, p.netloc, p.path, "", ""))


def download(url, path):
    """Cache local atomique; ne journalise jamais les parametres d'URL."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size:
        return path
    temporary = path.with_suffix(path.suffix + ".part")
    request = Request(url, headers={"User-Agent": "CitizHelper-Hackathon/1.0"})
    with urlopen(request, timeout=180) as response, temporary.open("wb") as target:
        while block := response.read(1024 * 1024):
            target.write(block)
    temporary.replace(path)
    return path


def acquire(config):
    raw = Path(config.workdir) / "raw"
    manifest = []
    urls = {k: v for k, v in SOURCES.items() if k != "gtfs_catalogue"}
    urls["communes.json"] = f"https://geo.api.gouv.fr/epcis/{config.epci}/communes?format=geojson&geometry=contour"
    for name, url in urls.items():
        download(url, raw / name)
        manifest.append({"source": name, "url": public_url(url),
                         "retrieved_at_utc": datetime.fromtimestamp((raw/name).stat().st_mtime, timezone.utc).isoformat()})
        print("Source disponible:", name, flush=True)
    gbfs = json.loads((raw / "citiz.json").read_text())
    feeds = gbfs["data"].get("feeds")
    if feeds is None:  # GBFS 2.x
        feeds = next(iter(gbfs["data"].values()))["feeds"]
    stations_url = next(f["url"] for f in feeds if f["name"] == "station_information")
    download(stations_url, raw / "stations.json")
    manifest.append({"source": "stations.json", "url": public_url(stations_url),
                     "gbfs_last_updated": str(gbfs.get("last_updated", ""))})
    # Le GTFS officiel TCL exige une authentification. Cette copie publique est
    # publiee comme ressource communautaire Google sur le catalogue national.
    if not (raw / "gtfs_public.zip").exists():
        with urlopen(SOURCES["gtfs_catalogue"], timeout=90) as response:
            page = response.read().decode("utf-8")
        links = re.findall(r'''href=["']([^"']+)''', page)
        links = [html.unescape(u) for u in links
                 if "gtech-transit-prod.apigee.net/" in u and "lyon_tcl.zip" in u]
        if not links:
            raise RuntimeError("GTFS TCL public absent. Fournir un GTFS autorise dans raw/gtfs_public.zip.")
        download(links[0], raw / "gtfs_public.zip")
    manifest.append({"source": "gtfs_public.zip", "url": SOURCES["gtfs_catalogue"],
                     "note": "Copie publique Google referencee par le PAN; dates de service verifiees"})
    return raw, pd.DataFrame(manifest).fillna("")


def prepare_population(raw):
    import geopandas as gpd
    communes = gpd.GeoDataFrame.from_features(json.loads((raw/"communes.json").read_text())["features"], crs=4326)
    communes = communes[["code", "nom", "geometry"]].rename(columns={"code": "commune_code", "nom": "commune"})
    # Lecture par morceaux, sans charger toute la population nationale en RAM.
    parts = []
    with zipfile.ZipFile(raw / "population.zip") as archive:
        with archive.open("carreaux_200m_met.csv") as stream:
            for chunk in pd.read_csv(stream, dtype={"lcog_geo": str}, chunksize=150_000):
                keep = chunk.lcog_geo.str.contains(r"(?:^|[,; ])69\d{3}(?:$|[,; ])", regex=True, na=False)
                parts.append(chunk.loc[keep].copy())
    population = pd.concat(parts, ignore_index=True)
    coordinates = population.idcar_200m.str.extract(r"N(\d+)E(\d+)").astype(float)
    if coordinates.isna().any().any():
        raise ValueError("Identifiants de grille non reconnus; projection non devinee.")
    points = gpd.GeoDataFrame(population, geometry=gpd.points_from_xy(coordinates[1]+100, coordinates[0]+100), crs=3035).to_crs(4326)
    cells = gpd.sjoin(points, communes, predicate="within", how="inner").drop(columns="index_right").reset_index(drop=True)
    if cells.idcar_200m.duplicated().any():
        raise ValueError("Cellules affectees a plusieurs communes.")
    projected = cells.to_crs(2154)
    cells["x"] = projected.geometry.x
    cells["y"] = projected.geometry.y
    cells["latitude"] = cells.geometry.y
    cells["longitude"] = cells.geometry.x
    cells = cells.rename(columns={"idcar_200m": "cell_id", "ind": "population", "men": "households"})
    age_columns = ["ind_18_24", "ind_25_39", "ind_40_54", "ind_55_64", "ind_65_79", "ind_80p"]
    cells["adults"] = cells[age_columns].sum(axis=1)
    cells["age_25_39"] = cells["ind_25_39"]
    cells["poor_households"] = cells["men_pauv"]
    cells = cells.loc[(cells.population > 0) & (cells.households > 0)].copy().reset_index(drop=True)
    # Le RP est agrégé à la commune dans cette V1. Pas de fausse precision IRIS.
    columns = ["COM", "P22_MEN", "P22_RP_VOIT1P", "P22_RP_VOIT1"]
    with zipfile.ZipFile(raw / "logement.zip") as archive:
        name = next(n for n in archive.namelist() if n.lower().startswith("base-ic") and n.lower().endswith(".csv"))
        with archive.open(name) as stream:
            housing = pd.read_csv(stream, sep=";", usecols=columns, dtype={"COM": str}, low_memory=False)
    for column in columns[1:]:
        housing[column] = pd.to_numeric(housing[column], errors="coerce")
    housing["COM"] = housing.COM.replace({str(69380+i): "69123" for i in range(1, 10)})
    housing = housing.groupby("COM")[columns[1:]].sum(min_count=1)
    housing["no_car_rate"] = (1 - housing.P22_RP_VOIT1P / housing.P22_MEN.replace(0, np.nan)).clip(0, 1)
    housing["one_car_rate"] = (housing.P22_RP_VOIT1 / housing.P22_MEN.replace(0, np.nan)).clip(0, 1)
    cells = cells.merge(housing[["no_car_rate", "one_car_rate"]], left_on="commune_code", right_index=True, how="left")
    cells["no_car_households_proxy"] = cells.households * cells.no_car_rate
    cells["license_holders_without_car"] = np.nan  # permis NON mesure
    cells["motorization_scale"] = "commune_RP2022_proxy"
    if cells.empty:
        raise ValueError("Aucune cellule dans le territoire choisi.")
    return cells.reset_index(drop=True), communes


def active_services(calendar, exceptions, day):
    stamp = day.strftime("%Y%m%d")
    weekday = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"][day.weekday()]
    ids = set()
    if not calendar.empty:
        valid = (calendar.start_date <= stamp) & (calendar.end_date >= stamp) & (calendar[weekday] == "1")
        ids = set(calendar.loc[valid, "service_id"])
    if not exceptions.empty:
        today = exceptions.loc[exceptions.date == stamp]
        ids |= set(today.loc[today.exception_type == "1", "service_id"])
        ids -= set(today.loc[today.exception_type == "2", "service_id"])
    return ids


def read_transit(raw, config):
    with zipfile.ZipFile(raw / "gtfs_public.zip") as archive:
        names = {Path(n).name: n for n in archive.namelist()}
        def read(name, columns=None, optional=False):
            if name not in names and optional:
                return pd.DataFrame()
            with archive.open(names[name]) as stream:
                return pd.read_csv(stream, dtype=str, usecols=columns)
        stops = read("stops.txt", ["stop_id", "stop_name", "stop_lat", "stop_lon"])
        trips = read("trips.txt", ["trip_id", "service_id"])
        calendar = read("calendar.txt", optional=True)
        exceptions = read("calendar_dates.txt", optional=True)
        events = read("stop_times.txt", ["trip_id", "stop_id", "departure_time"])
        frequencies = read("frequencies.txt", optional=True)
    day = date.fromisoformat(config.reference_date)
    weekday = day + timedelta(days=(1-day.weekday()) % 7)
    saturday = day + timedelta(days=(5-day.weekday()) % 7)
    times = events.departure_time.str.split(":", expand=True)
    hours = pd.to_numeric(times[0], errors="raise") + pd.to_numeric(times[1], errors="raise")/60
    events["hour"] = hours
    frequency_events = pd.DataFrame()
    if not frequencies.empty:
        for key in ["start_time", "end_time"]:
            t = frequencies[key].str.split(":", expand=True).astype(int)
            frequencies[key+"_sec"] = t[0]*3600+t[1]*60+t[2]
        frequencies["headway_secs"] = pd.to_numeric(frequencies.headway_secs, errors="raise")
        if (frequencies.headway_secs <= 0).any():
            raise ValueError("Intervalle GTFS non positif.")
        templates = events.loc[events.trip_id.isin(frequencies.trip_id)].copy()
        first = templates.groupby("trip_id").hour.min()
        templates["offset_sec"] = (templates.hour-templates.trip_id.map(first))*3600
        frequency_events = templates.merge(frequencies, on="trip_id").merge(trips, on="trip_id")
        events = events.loc[~events.trip_id.isin(frequencies.trip_id)]
    metrics = []
    for label, service_day, low, high in [("weekday", weekday, 6, 20), ("evening", weekday, 20, 24), ("weekend", saturday, 6, 24)]:
        services = active_services(calendar, exceptions, service_day)
        if not services:
            raise ValueError(f"GTFS sans service actif le {service_day}: classement courant impossible.")
        active_trips = set(trips.loc[trips.service_id.isin(services), "trip_id"])
        selected = events.loc[events.trip_id.isin(active_trips) & events.hour.ge(low) & events.hour.lt(high)]
        count = selected.drop_duplicates(["trip_id", "stop_id"]).groupby("stop_id").size()/(high-low)
        if not frequency_events.empty:
            f = frequency_events.loc[frequency_events.service_id.isin(services)].copy()
            lower = np.maximum(f.start_time_sec, low*3600-f.offset_sec)
            upper = np.minimum(f.end_time_sec, high*3600-f.offset_sec)
            f["count"] = np.maximum(0, np.ceil((upper-f.start_time_sec)/f.headway_secs)-np.maximum(0, np.ceil((lower-f.start_time_sec)/f.headway_secs)))
            count = count.add(f.groupby("stop_id")["count"].sum()/(high-low), fill_value=0)
        stops[f"departures_per_hour_{label}"] = stops.stop_id.map(count).fillna(0)
        metrics.append({"scenario": label, "service_date": service_day.isoformat(), "window": f"{low}-{high}"})
    for column in ["stop_lat", "stop_lon"]:
        stops[column] = pd.to_numeric(stops[column], errors="coerce")
    return stops.dropna(subset=["stop_lat", "stop_lon"]), pd.DataFrame(metrics)


def enrich_access(cells, stops, raw, config):
    import geopandas as gpd
    from pyproj import Transformer
    transform = Transformer.from_crs(4326, 2154, always_xy=True)
    stops = stops.copy()
    stops["x"], stops["y"] = transform.transform(stops.stop_lon.to_numpy(), stops.stop_lat.to_numpy())
    nearest = cKDTree(stops[["x", "y"]].to_numpy())
    neighbours = nearest.query_ball_point(cells[["x", "y"]].to_numpy(), config.radius_m)
    cells["transit_stop_count"] = [len(n) for n in neighbours]
    for label in ["weekday", "evening", "weekend"]:
        values = stops[f"departures_per_hour_{label}"].to_numpy()
        # Meilleur arret proche; evite de compter plusieurs fois le meme bus.
        cells[f"best_departures_per_hour_{label}"] = [max(values[n], default=0) for n in neighbours]
    cells["transit_gap"] = (
        .50*(1-np.minimum(cells.best_departures_per_hour_weekday/4, 1))
        + .25*(1-np.minimum(cells.best_departures_per_hour_evening/2, 1))
        + .25*(1-np.minimum(cells.best_departures_per_hour_weekend/2, 1)))
    stations = pd.DataFrame(json.loads((raw / "stations.json").read_text())["data"]["stations"])
    stations = stations.drop_duplicates("station_id").copy()
    stations["x"], stations["y"] = transform.transform(stations.lon.to_numpy(), stations.lat.to_numpy())
    distances, _ = cKDTree(stations[["x", "y"]].to_numpy()).query(cells[["x", "y"]].to_numpy())
    cells["nearest_citiz_m"] = distances
    cells["citiz_covered"] = distances <= config.radius_m
    cells["need_heat_weight"] = cells.no_car_households_proxy * (.25+.75*cells.transit_gap) * (~cells.citiz_covered)
    # La couche transport reste un diagnostic de l'offre, pas une frequentation.
    stops["nearby_population"] = [float(cells.iloc[n].population.sum()) for n in cKDTree(cells[["x", "y"]]).query_ball_point(stops[["x", "y"]], config.radius_m)]
    stations["nearby_population"] = [float(cells.iloc[n].population.sum()) for n in cKDTree(cells[["x", "y"]]).query_ball_point(stations[["x", "y"]], config.radius_m)]
    return cells, stops, stations


def score_benefit(population, no_car_households, transit_gap, poverty_rate, age_share, config):
    """References fixes et poids explicites; pas de ruralite ni de permis invente."""
    terms = {
        "points_no_car": config.no_car_weight*np.clip(no_car_households/config.target_households_anchor, 0, 1),
        "points_transit_gap": config.transit_weight*np.clip(transit_gap, 0, 1),
        "points_new_population": config.population_weight*np.clip(population/config.population_anchor, 0, 1),
        "points_equity": config.equity_weight*np.clip(poverty_rate/.4, 0, 1),
        "points_age": config.age_bonus_max*np.clip(age_share/.4, 0, 1),
    }
    terms["benefit_score"] = sum(terms.values())
    return terms


def cost_penalty(cost, config):
    if not np.isfinite(cost) or cost < 0:
        raise ValueError("Cout manquant ou negatif: le site ne peut pas etre optimise.")
    return config.max_cost_penalty*min(cost/config.cost_reference_eur, 2)


def score_mobility_need(population, target, gap, poverty, age, config):
    """Besoin lie a une cible substantielle ET au manque de TC, sans bonus rural."""
    gap=np.clip(gap,0,1)
    support=np.clip(target/config.target_households_anchor,0,1)
    terms=score_benefit(population,target,gap,poverty,age,config)
    for key in ['points_no_car','points_new_population']:
        terms[key]*=gap
    terms['points_transit_gap']=config.transit_weight*np.clip(target*gap/config.target_households_anchor,0,1)
    for key in ['points_equity','points_age']:
        terms[key]*=gap*support
    terms['benefit_score']=sum(v for k,v in terms.items() if k!='benefit_score')
    terms['transit_coverage_penalty']=config.transit_coverage_penalty*(1-gap)
    terms['mobility_need_score']=np.maximum(0,terms['benefit_score']-terms['transit_coverage_penalty'])
    return terms


def evaluate_site(cells, indices, config, cost):
    group = cells.iloc[indices]
    population = float(group.population.sum())
    if population <= 0:
        return None
    if group.no_car_rate.isna().any():
        return None  # absence de donnees: signaler, ne pas convertir en zero
    households = float(group.households.sum())
    no_car = float(group.no_car_households_proxy.sum())
    target_column='licensed_adults_no_car_estimate' if 'licensed_adults_no_car_estimate' in group else 'no_car_households_proxy'
    if group[target_column].isna().any(): return None
    target=float(group[target_column].sum())
    gap = float((group[target_column]*group.transit_gap).sum()/max(target,1e-12))
    poverty = float(group.poor_households.sum()/max(households, 1))
    age = float(group.age_25_39.sum()/max(group.adults.sum(), 1))
    terms = score_mobility_need(population, target, gap, poverty, age, config)
    penalty = cost_penalty(cost, config)
    return {"new_population": population, "new_no_car_households_proxy": no_car,
            "transit_gap": gap, "poverty_rate": poverty, "share_25_39_adults": age,
            **terms, "cost_penalty": penalty,
            "priority_score": max(0., terms["mobility_need_score"]-penalty),
            "cost_per_new_resident_eur": cost/population}


def rank_and_select(cells, config, cost_overrides=None):
    config.validate()
    overrides = {} if cost_overrides is None else cost_overrides
    xy = cells[["x", "y"]].to_numpy()
    areas = cKDTree(xy).query_ball_point(xy, config.radius_m)
    covered = cells.citiz_covered.astype('boolean')
    available = ~covered.fillna(True).to_numpy(dtype=bool)
    # Le centre propose doit lui aussi etre hors de la couverture existante.
    # Sans cette regle, ses voisins non couverts pouvaient le rendre attractif.
    site_eligible = available.copy()
    if 'nearest_citiz_m' in cells:
        distance = pd.to_numeric(cells.nearest_citiz_m, errors='coerce')
        available &= distance.notna().to_numpy()
        site_eligible &= distance.gt(config.radius_m).fillna(False).to_numpy()
    initially_available = available.copy()
    default_cost = config.setup_eur + config.vehicle_eur + config.horizon_years*config.annual_operation_eur
    # Matrice creuse: un seul calcul vectorise par iteration de selection.
    # Aucun cluster distribue; toutes les operations restent en Python/NumPy.
    from scipy.sparse import csr_matrix
    rows = np.repeat(np.arange(len(cells)), [len(a) for a in areas])
    columns = np.concatenate(areas).astype(int)
    matrix = csr_matrix((np.ones(len(columns)), (rows, columns)), shape=(len(cells), len(cells)))
    identifiers = cells.cell_id.to_numpy()
    costs = np.array([overrides.get(i, default_cost) for i in identifiers], dtype=float)
    if np.any(costs < 0):
        raise ValueError("Cout negatif interdit.")
    penalty = config.max_cost_penalty*np.minimum(costs/config.cost_reference_eur, 2)
    def metrics(mask):
        def total(column):
            return np.asarray(matrix @ (cells[column].fillna(0).to_numpy()*mask)).ravel()
        population = total("population")
        households = total("households")
        no_car = total("no_car_households_proxy")
        target = total('licensed_adults_no_car_estimate') if 'licensed_adults_no_car_estimate' in cells else no_car
        target_column='licensed_adults_no_car_estimate' if 'licensed_adults_no_car_estimate' in cells else 'no_car_households_proxy'
        gap = np.asarray(matrix @ (cells[target_column].fillna(0).to_numpy()*cells.transit_gap.to_numpy()*mask))/np.maximum(target,1e-12)
        poverty = total("poor_households")/np.maximum(households, 1)
        age = total("age_25_39")/np.maximum(total("adults"), 1)
        terms = score_mobility_need(population, target, gap, poverty, age, config)
        missing = np.asarray(matrix @ (cells.no_car_rate.isna().to_numpy()*mask)).ravel() > 0
        if 'licensed_adults_no_car_estimate' in cells:
            missing |= np.asarray(matrix @ (cells.licensed_adults_no_car_estimate.isna().to_numpy()*mask)).ravel() > 0
        eligible = (population > 0) & ~missing & np.isfinite(costs)
        return pd.DataFrame({"new_population": population, "new_no_car_households_proxy": no_car,
            'new_licensed_adults_no_car_estimate':target if 'licensed_adults_no_car_estimate' in cells else np.nan,
            "transit_gap": gap, "poverty_rate": poverty, "share_25_39_adults": age,
            **terms, "cost_penalty": penalty,
            "priority_score": np.where(eligible, np.maximum(0, terms["mobility_need_score"]-penalty), np.nan),
            "cost_per_new_resident_eur": costs/np.maximum(population, 1)})
    base = cells[["cell_id", "commune_code", "commune", "latitude", "longitude", "x", "y"]].rename(columns={"cell_id": "candidate_id"}).reset_index(drop=True)
    base["cost_total_eur"] = costs
    base["horizon_years"] = config.horizon_years
    base["cost_status"] = ["inconnu" if not np.isfinite(c) else "source_renseignee_a_verifier" for c in costs]
    base['cost_source']=config.cost_source
    base['budget_source']=config.budget_source
    base["site_status"] = "centre_carreau_parking_non_verifie"
    base['nearest_existing_citiz_m'] = cells.nearest_citiz_m.to_numpy() if 'nearest_citiz_m' in cells else np.nan
    base['existing_citiz_exclusion_m'] = config.radius_m
    base["license_status"] = "non_mesure"
    base["motorization_scale"] = "commune_RP2022_proxy"
    base["population_in_radius"] = np.asarray(matrix @ cells.population.to_numpy())
    candidates = pd.concat([base, metrics(available)], axis=1)
    candidates = candidates.loc[site_eligible & candidates.new_population.gt(0)].copy()
    usable_areas = {i: np.asarray(a, dtype=int) for i, a in zip(identifiers, areas)}
    candidates = candidates.sort_values(["priority_score", "candidate_id"], ascending=[False, True]).reset_index(drop=True)
    candidates["rank"] = np.arange(1, len(candidates)+1)
    candidates["age_band"] = "25-39 (proxy de la cible 25-35)"
    chosen, spent = [], 0.
    taken = np.zeros(len(cells), dtype=bool)
    for _ in range(config.max_stations):
        marginal = metrics(available)
        feasible = site_eligible & ~taken & (costs+spent <= config.budget_eur) & marginal.priority_score.gt(0).to_numpy()
        if not feasible.any():
            break
        choices = marginal.loc[feasible].copy()
        choices["cost"] = costs[feasible]
        i = choices.sort_values(["priority_score", "cost"], ascending=[False, True], kind="stable").index[0]
        candidate = base.iloc[i].to_dict()
        original = candidates.loc[candidates.candidate_id == candidate["candidate_id"]].iloc[0]
        candidate["rank"] = int(original["rank"])
        candidate["age_band"] = original["age_band"]
        value = marginal.iloc[i].to_dict()
        indices = usable_areas[candidate["candidate_id"]]
        fresh = indices[available[indices]]
        spent += costs[i]
        chosen.append({**candidate, **value, "selection_order": len(chosen)+1, "cumulative_cost_eur": spent})
        available[fresh] = False
        taken[i] = True
    selection = pd.DataFrame(chosen, columns=list(candidates.columns)+["selection_order", "cumulative_cost_eur"])
    cells["added_by_scenario"] = initially_available & ~available
    cells["covered_after_scenario"] = covered | cells.added_by_scenario
    return candidates, selection, usable_areas


def build_map(cells, stations, candidates, selection, output):
    import folium
    from folium.plugins import HeatMap, MarkerCluster
    import geopandas as gpd
    centre = [float(cells.latitude.mean()), float(cells.longitude.mean())]
    m = folium.Map(location=centre, zoom_start=11, tiles="OpenStreetMap", control_scale=True)
    heat = folium.FeatureGroup(name="Chaleur: besoin Citiz estime", show=True)
    values = cells.dropna(subset=["need_heat_weight"])
    scale = max(float(values.need_heat_weight.quantile(.95)), 1)
    HeatMap([[r.latitude, r.longitude, min(r.need_heat_weight/scale, 1)] for r in values.itertuples() if r.need_heat_weight > 0],
            radius=16, blur=12, min_opacity=.2).add_to(heat)
    heat.add_to(m)
    # Carreaux colores pour la lecture analytique; geometrie originale 200m.
    from shapely.geometry import box
    from branca.colormap import LinearColormap
    identifiers = cells.cell_id.str.extract(r"N(\d+)E(\d+)").astype(float)
    squares = gpd.GeoDataFrame(cells[["cell_id", "commune", "population", "transit_gap", "no_car_rate", "need_heat_weight", "citiz_covered"]].copy(),
                              geometry=[box(e, n, e+200, n+200) for n, e in identifiers.to_numpy()], crs=3035).to_crs(4326)
    squares = squares.astype({"citiz_covered": str})
    palette = LinearColormap(["#dcecf4", "#e8c26e", "#d54c3f"], vmin=0, vmax=scale)
    folium.GeoJson(json.loads(squares.to_json()), name="Analyse: carreaux 200m", show=False,
        style_function=lambda f: {"fillColor": palette(min(f["properties"]["need_heat_weight"] or 0, scale)), "color": "#ffffff", "weight": .2, "fillOpacity": .65},
        tooltip=folium.GeoJsonTooltip(fields=["commune", "population", "transit_gap", "no_car_rate", "citiz_covered"],
                                     aliases=["Commune", "Population Filosofi", "Deficit TC (0-1)", "Sans voiture: taux COMMUNAL", "Citiz proche"])).add_to(m)
    cluster = MarkerCluster(name="Stations Citiz existantes", show=True).add_to(m)
    for row in stations.itertuples():
        name = row.name if isinstance(row.name, str) else " / ".join(n.get("text", "") for n in row.name)
        folium.Marker([row.lat, row.lon], tooltip=html.escape(name),
                      popup=f"{html.escape(name)}<br>Habitants proches: {row.nearby_population:,.0f}", icon=folium.Icon(color="blue", icon="car", prefix="fa")).add_to(cluster)
    candidates_layer = folium.FeatureGroup(name="Top 100 secteurs a etudier", show=False).add_to(m)
    def add_candidate(row, layer, selected=False):
        text = (f"<b>{html.escape(row.commune)}</b><br>Score: {row.priority_score:.1f}/100"
                f"<br>Habitants nouvellement proches: {row.new_population:,.0f}"
                f"<br>Menages sans voiture (proxy communal): {row.new_no_car_households_proxy:,.0f}"
                f"<br>Permis: non mesure<br>Cout {row.cost_total_eur:,.0f} EUR / {row.horizon_years} ans (hypothese)"
                f"<br>Bonus age 25-39: {row.points_age:.1f}/5"
                "<br>Rayon aerien 400m; parking a verifier")
        folium.CircleMarker([row.latitude, row.longitude], radius=8 if selected else 5,
                            color="#087f70" if selected else "#ba6a20", fill=True, fill_opacity=.9,
                            tooltip=f"{row.commune}: {row.priority_score:.1f}", popup=folium.Popup(text, max_width=340)).add_to(layer)
    for row in candidates.head(100).itertuples():
        add_candidate(row, candidates_layer)
    chosen_layer = folium.FeatureGroup(name="Selection sous budget (scenario)", show=True).add_to(m)
    for row in selection.itertuples():
        add_candidate(row, chosen_layer, True)
    folium.LayerControl(collapsed=False).add_to(m)
    m.save(str(output))
    return m


def export_results(result, config):
    output = Path(config.workdir)/"output"
    output.mkdir(parents=True, exist_ok=True)
    summary = pd.DataFrame(result["cells"].drop(columns="geometry")).copy()
    summary["population_covered"] = summary.population * summary.citiz_covered
    summary["population_added"] = summary.population * summary.added_by_scenario
    commune_kpis = summary.groupby(["commune_code", "commune"], as_index=False).agg(
        population=("population", "sum"), population_covered=("population_covered", "sum"),
        population_added=("population_added", "sum"), households=("households", "sum"),
        no_car_households_proxy=("no_car_households_proxy", "sum"))
    tables = {
        "citiz_v1_coverage_cells": pd.DataFrame(result["cells"].drop(columns="geometry")),
        "citiz_v1_candidates": result["candidates"],
        "citiz_v1_selection": result["selection"],
        "citiz_v1_stations": result["stations"][["station_id", "lat", "lon", "nearby_population"]].astype({"station_id": str}),
        "citiz_v1_transit_stops": result["stops"],
        "citiz_v1_sources": result["sources"],
        "citiz_v1_service_dates": result["service_dates"],
        "citiz_v1_commune_kpis": commune_kpis,
    }
    for name, table in tables.items():
        table.to_csv(output/f"{name}.csv", index=False)
    (output/"config.json").write_text(json.dumps(asdict(config), indent=2), encoding="utf-8")
    result["map"] = build_map(result["cells"], result["stations"], result["candidates"], result["selection"], output/"citiz_map.html")
    result["tables"] = tables
    result["output"] = output
    return result


def run(config=None):
    config = config or Config()
    config.validate()
    raw, sources = acquire(config)
    print("Preparation de la population et de la motorisation...", flush=True)
    cells, communes = prepare_population(raw)
    print(len(cells), "carreaux habites;", round(cells.population.sum()), "habitants Filosofi", flush=True)
    stops, service_dates = read_transit(raw, config)
    cells, stops, stations = enrich_access(cells, stops, raw, config)
    print("Calcul des candidats et du gain marginal sous budget...", flush=True)
    overrides_path = Path(config.workdir)/"cost_overrides.csv"
    overrides = None
    if overrides_path.exists():
        costs = pd.read_csv(overrides_path)
        if costs.candidate_id.duplicated().any():
            raise ValueError("Couts dupliques pour un candidat.")
        overrides = dict(zip(costs.candidate_id, costs.cost_total_eur))
    candidates, selection, areas = rank_and_select(cells, config, overrides)
    result = export_results(dict(cells=cells, communes=communes, stops=stops, stations=stations,
                                 sources=sources, service_dates=service_dates,
                                 candidates=candidates, selection=selection), config)
    print(f"TERMINE: {len(candidates)} secteurs, {len(selection)} stations dans le scenario.", flush=True)
    print("Permis non mesures; couts hypothetiques; age 25-39; rayon aerien 400m.", flush=True)
    return result




In [ ]:
"""Tables d'analyse explicables : geographie, besoins, score et qualite."""
from pathlib import Path
import html
import numpy as np
import pandas as pd


def _columns(data,spec):
    frame=pd.DataFrame(index=data.index)
    for group,label,name in spec:
        frame[(group,label)]=data[name] if name in data else np.nan
    frame.columns=pd.MultiIndex.from_tuples(frame.columns)
    return frame.reset_index(drop=True)


def build_analysis_tables(france,ranked,cells,metrics_fn):
    local=cells is not None
    scored=ranked.copy()
    population=cells.copy() if local else ranked.copy()
    if local and 'cell_id' in population: population=population.drop_duplicates('cell_id')
    metadata=france.get('overview',pd.DataFrame())
    for frame_name in ['scored','population']:
        frame=scored if frame_name=='scored' else population
        if 'commune_code' in metadata and 'commune_code' in frame:
            missing=[c for c in ['region','region_name','department','department_name','commune'] if c not in frame and c in metadata]
            if missing: frame=frame.merge(metadata[['commune_code',*missing]].drop_duplicates('commune_code'),on='commune_code',how='left',validate='many_to_one')
        if frame_name=='scored': scored=frame
        else: population=frame
    ordered=scored.sort_values('priority_score',ascending=False,na_position='last')
    geography=[('Territoire','Code INSEE','commune_code'),('Territoire','Commune','commune'),('Territoire','Departement','department_name')]
    target='new_licensed_adults_no_car_estimate' if local else 'uncovered_licensed_adults_no_car_estimate'
    people='new_population' if local else 'uncovered_population'
    ranking=_columns(ordered,geography+[
        ('Besoin','Permis sans vehicule hors Citiz',target),
        ('Besoin','Habitants non couverts dans le rayon (non additif)' if local else 'Habitants hors Citiz',people),
        ('Transports','Manque TC de la cible (0-1)','transit_gap' if local else 'target_transit_gap'),
        ('Decision','Score /100','priority_score'),('Qualite','Statut permis','license_status'),
        ('Qualite','Statut du modele','evidence_status')])
    score_spec=geography[:2]+[
        ('Points de besoin','Cible permis','points_no_car'),
        ('Points de besoin','Eloignement TC','points_transit_gap' if local else 'points_transit'),
        ('Points de besoin','Population','points_new_population' if local else 'points_population'),
        ('Points de besoin','Social','points_equity'),('Points de besoin','Age','points_age'),
        ('Penalites','TC deja proches','transit_coverage_penalty')]
    if 'cost_penalty' in scored and scored.cost_penalty.notna().any():
        score_spec += [('Penalites','Couts documentes','cost_penalty'),('Finances','Cout sur horizon EUR','cost_total_eur'),('Finances','Reference','cost_source')]
    scores=_columns(ordered,score_spec+[('Resultat','Score /100','priority_score')])
    tables={'classement':ranking,'decomposition':scores}
    for key,title in [('region','regions'),('department','departements')]:
        if key not in population: continue
        records=[]
        for code,group in population.groupby(key,dropna=False,sort=True):
            measures=metrics_fn(group,group if local else None)
            group_scores=scored.loc[scored[key].eq(code)] if key in scored and pd.notna(code) else scored.iloc[:0]
            name_col=key+'_name'
            labels=group[name_col].dropna() if name_col in group else pd.Series(dtype=str)
            known=group_scores.priority_score.dropna()
            records.append({('Territoire','Code'):str(code) if pd.notna(code) else 'Non renseigne',
                ('Territoire','Nom'):str(labels.iloc[0]) if len(labels) else '',
                ('Population','Etudiee'):measures['population'],('Cible','Permis hors Citiz estime'):measures['target'],
                ('Transports','A plus de 400 m des arrets %'):measures['far_share'],
                ('Citiz','A moins de 400 m %'):measures['citiz_share'],
                ('Classement','Candidats' if local else 'Communes'):len(group_scores),
                ('Classement','Scores calculables'):len(known),('Classement','Mediane /100'):known.median(),
                ('Classement','Maximum /100'):known.max(),('Classement','Score >=40'):int(known.ge(40).sum())})
        table=pd.DataFrame(records)
        if len(table): table.columns=pd.MultiIndex.from_tuples(table.columns)
        tables[title]=table
    bins=pd.cut(scored.priority_score,[-np.inf,20,40,60,np.inf],labels=['0 a <20','20 a <40','40 a <60','60 et plus'],right=False)
    classes=bins.astype(object).fillna('Non calculable')
    distribution=classes.value_counts().reindex(['0 a <20','20 a <40','40 a <60','60 et plus','Non calculable'],fill_value=0)
    tables['distribution']=pd.DataFrame({'Classe de score':distribution.index,'Nombre de lignes':distribution.values,
        'Part des lignes %':100*distribution.values/max(len(scored),1)})
    quality=[]
    for column in ['priority_score',target,'population_far_stops','uncovered_population','license_coefficient','no_car_rate','cost_total_eur','cost_source']:
        if column not in scored: continue
        values=scored[column]
        missing=values.isna() | values.astype(str).str.strip().eq('')
        quality.append(dict(Variable=column,Lignes=len(values),Renseignees=int((~missing).sum()),Manquantes=int(missing.sum()),
            Part_manquante_pct=100*missing.mean() if len(values) else np.nan))
    tables['qualite']=pd.DataFrame(quality)
    tables['sources']=france.get('sources',pd.DataFrame()).copy()
    return tables


def analysis_report(tables,local=False):
    descriptions={
        'classement':('Classement detaille','Tri par score decroissant. '+('Les rayons se recouvrent : leurs gains ne s’additionnent pas.' if local else 'Les volumes portent sur les habitants hors couverture Citiz connue.')),
        'decomposition':('Decomposition et penalites','Points du scenario courant. Une colonne financiere apparait seulement si elle est renseignee.'),
        'regions':('Comparaison des regions','Pourcentages ponderes par la population renseignee. Mediane et maximum des scores, sans somme de scores.'),
        'departements':('Comparaison des departements','Population et cible agregees sans double comptage des carreaux. Les manquants restent exclus des denominateurs.'),
        'distribution':('Distribution des priorites','Effectifs et parts des lignes du classement. Les scores absents sont isoles.'),
        'qualite':('Completude des donnees','Les pourcentages portent sur les lignes, pas sur la population. Un champ rempli peut rester une estimation.'),
        'sources':('Inventaire et provenance','Sources disponibles et statuts de collecte. Un inventaire incomplet ne prouve pas une absence de service.')}
    result='<h3>Analyse du territoire</h3><p>Les tableaux suivent les filtres et les poids courants. Horaires, reservations et disponibilite des vehicules ne sont pas observes.</p>'
    for key,frame in tables.items():
        title,description=descriptions[key]
        limit=50 if key in ['classement','decomposition'] else 120
        subset=frame.head(limit)
        table=subset.to_html(index=False,escape=True,na_rep='Non renseigne',border=0,float_format=lambda x:f'{x:,.2f}'.replace(',',' '))
        text=f'{len(frame):,} lignes'.replace(',',' ')
        if len(frame)>limit: text+=f' · {limit} affichees · export CSV complet disponible'
        result+=(f'<details class="cb-analysis-table"{" open" if key=="classement" else ""}><summary>{html.escape(title)} <span>{text}</span></summary>'
                 f'<p>{html.escape(description)}</p><div class="cb-table-scroll">{table}</div></details>')
    return result


def export_analysis_tables(tables,folder,prefix='analyse_'):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    for name,frame in tables.items():
        data=frame.copy()
        if isinstance(data.columns,pd.MultiIndex): data.columns=[' / '.join(str(x) for x in column) for column in data.columns]
        data.to_csv(folder/(prefix+name+'.csv'),index=False,encoding='utf-8')


In [ ]:
"""Interface interactive Fabric: widgets -> recalcul Python -> nouvelle carte.

Sans Output/interact, non pris en charge par Fabric. Les widgets HTML sont mis
a jour en place. Une session Python ouverte est necessaire pour recalculer.
"""
from pathlib import Path
from dataclasses import asdict
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import html, json
import numpy as np
import pandas as pd
from pyproj import Transformer
try:
    from citiz_france import FranceScenario, france_rank, STUDY_REGIONS, STUDY_DEPARTMENTS, sphere_xyz, nearest_metres
except ImportError:
    pass  # definitions deja chargees dans le notebook Fabric
try:
    from mobility_metrics import annotate_evidence, mobility_panel_html
except ImportError:
    pass
try:
    from analysis_views import build_analysis_tables, analysis_report, export_analysis_tables
except ImportError:
    pass


PRIORITY_COLORS=[(20,'#2878a5','Faible · 0 a <20'),(40,'#d7aa28','Moderee · 20 a <40'),
                 (60,'#e07732','Elevee · 40 a <60'),(101,'#b52b42','Tres elevee · 60 a 100')]


def priority_color(score):
    if not np.isfinite(score): return '#77838c'
    return next(color for upper,color,_ in PRIORITY_COLORS if score<upper)


MAP_THEMES={
    'priority':('Priorites d implantation','Score de besoin /100, avec les poids du scenario. Centres de communes ou de carreaux a etudier.'),
    'target':('Permis et absence de vehicule','Personnes estimees avec permis B et aucun vehicule disponible, hors couverture Citiz.'),
    'transit':('Eloignement des arrets','Part de population a plus de 400 m a vol d oiseau d un arret publie. Proxy spatial : les horaires, frequences et destinations ne sont pas mesures.'),
    'citiz':('Couverture Citiz','Population estimee a moins de 400 m a vol d oiseau d une station connue. L inventaire peut etre incomplet ; pas une mesure de disponibilite des voitures.'),
    'evidence':('Solidite des donnees','Violet : estimations et proxys a valider. Gris : donnees insuffisantes. Aucun indicateur de precision statistique invente.')}


HEAT_GRADIENT={0.0:'#245db5',.25:'#22a7c7',.5:'#f0d447',.75:'#ed8537',1.0:'#bb2345'}


def heat_intensities(weights):
    """Echelle log relative au perimetre ; zeros et manquants sans chaleur."""
    values=np.asarray(weights,dtype=float)
    positive=np.isfinite(values)&(values>0)
    result=np.zeros(len(values),dtype=float)
    if positive.any():
        logged=np.log1p(values[positive])
        result[positive]=np.minimum(logged/max(float(np.quantile(logged,.99)),1e-12),1)
    return result


def add_known_stations(m,stations,show=True):
    import folium
    from folium.plugins import FastMarkerCluster
    layer=folium.FeatureGroup(name='Stations Citiz connues (bleu)',show=show).add_to(m)
    FastMarkerCluster([[float(r.latitude),float(r.longitude),html.escape(str(r.name))] for r in stations.itertuples()],control=False,
        icon_create_function="function(c){return L.divIcon({html:'<div style=\"background:#2367a2;color:white;border:2px solid white;border-radius:50%;width:28px;height:28px;line-height:28px;text-align:center\">'+c.getChildCount()+'</div>',className:'citiz-existing-cluster',iconSize:[32,32]});}",
        callback="function(row){return L.marker([row[0],row[1]],{icon:L.divIcon({html:'<span style=\"background:#2367a2;color:white;border:2px solid white;border-radius:4px;padding:2px 4px;font:bold 12px Arial\">C</span>',className:'citiz-existing',iconSize:[22,22]})}).bindPopup('Station Citiz connue<br>'+row[2]);}").add_to(layer)
    return layer


def fit_study_bounds(m,data):
    located=data.dropna(subset=['latitude','longitude'])
    if located.empty: return
    bounds=[[float(located.latitude.min()),float(located.longitude.min())],
            [float(located.latitude.max()),float(located.longitude.max())]]
    m._citiz_study_bounds=bounds
    m.fit_bounds(bounds,max_zoom=12,padding=[24,24])
    import folium
    name=m.get_name()
    # Le conteneur Fabric et le panneau BI peuvent changer de largeur apres rendu.
    m.get_root().script.add_child(folium.Element(f"window.addEventListener('load',()=>setTimeout(()=>{{{name}.invalidateSize();{name}.fitBounds({json.dumps(bounds)},{{maxZoom:12,padding:[24,24]}});}},250));"))


def map_availability(ranked,cells,theme):
    data=ranked if cells is None or theme in ['priority','evidence'] else cells
    located=data.latitude.notna()&data.longitude.notna()
    values=pd.Series(1.,index=data.index) if theme=='evidence' else data.priority_score if theme=='priority' else thematic_values(ranked,cells,theme).map_value
    known=located&np.isfinite(values)
    return dict(total=len(data),geocoded=int(located.sum()),available=int(known.sum()))


def thematic_values(ranked,cells,theme):
    """Variables cartographiees distinctes du score. Ne modifie pas le classement."""
    if theme not in MAP_THEMES: raise ValueError('Carte inconnue : '+str(theme))
    data=(ranked if cells is None else cells).copy()
    def numeric(name):
        return pd.to_numeric(data[name],errors='coerce') if name in data else pd.Series(np.nan,index=data.index)
    if cells is None:
        target=numeric('uncovered_licensed_adults_no_car_estimate')
        population=numeric('population')
        transit=(numeric('population_far_stops')/population.where(population>0)).clip(0,1)
        coverage=(1-numeric('uncovered_population')/population.where(population>0)).clip(0,1)
    else:
        target=numeric('licensed_adults_no_car_estimate')
        covered=data['citiz_covered'] if 'citiz_covered' in data else None
        if covered is not None: target=target.where(~covered.fillna(True).astype(bool),0).where(covered.notna())
        population=numeric('population')
        distance=numeric('nearest_stop_m')
        transit=distance.gt(400).astype(float).where(distance.notna())
        coverage=data['citiz_covered'].astype(float) if 'citiz_covered' in data else numeric('missing_coverage')
    data['map_value']=target.clip(lower=0) if theme=='target' else 100*(transit if theme=='transit' else coverage)
    data['map_order']=population if theme in ['transit','citiz'] else target
    data['map_heat_weight']=target if theme=='target' else target*transit if theme=='transit' else population*coverage
    return data.replace([np.inf,-np.inf],np.nan)


def _map_travel_controls(m):
    import folium
    uid=m.get_name()
    m.get_root().header.add_child(folium.Element('''<style>
    .leaflet-tile-pane{filter:grayscale(1) contrast(.94) brightness(1.04)}
    .leaflet-container{font:12px "Segoe UI",Arial,sans-serif;background:#edf1f2}
    .leaflet-control-layers,.leaflet-bar{border:1px solid #dce4e6!important;border-radius:7px!important;box-shadow:0 2px 8px #183f4812!important}
    .leaflet-popup-content{font:12px "Segoe UI",Arial,sans-serif;line-height:1.65}
    </style>'''))
    m.get_root().script.add_child(folium.Element(f"document.getElementById('{uid}').appendChild(document.getElementById('{uid}_legend'));"))


def _indicator_map(france,ranked,cells,theme):
    import folium
    from folium.plugins import HeatMap,FastMarkerCluster,Fullscreen
    data=thematic_values(ranked,cells,theme).dropna(subset=['latitude','longitude'])
    title,description=MAP_THEMES[theme]
    m=folium.Map(location=[46.7,2.3],zoom_start=5,zoom_snap=.1,tiles='OpenStreetMap',height=540,control_scale=True,prefer_canvas=True)
    m.get_root().height='540px'; Fullscreen(position='topleft').add_to(m)
    folium.map.CustomPane('priority_points',z_index=625,pointer_events=True).add_to(m)
    valid=data.loc[data.map_value.notna()].copy()
    if theme=='target':
        colors=['#d4c5ed','#b09bd8','#8363b6','#563487']
        bounds=[100,500,2000,np.inf] if cells is None else [1,5,20,np.inf]
        labels=['<100','100 a <500','500 a <2000','2000 et plus'] if cells is None else ['<1','1 a <5','5 a <20','20 et plus']
        unit='personnes estimees'
        heat=folium.FeatureGroup(name='Concentration de la cible estimee',show=True).add_to(m)
        weight=valid.map_heat_weight.clip(lower=0)
        intensity=heat_intensities(weight)
        HeatMap([[float(a),float(b),float(w)] for a,b,w in zip(valid.latitude,valid.longitude,intensity) if w>0],
                radius=16,blur=12,min_opacity=.12,max_zoom=5 if cells is None else 9,gradient=HEAT_GRADIENT).add_to(heat)
    elif theme=='transit':
        colors=['#2878a5','#e3c35b','#ed9046','#b52b42']; bounds=[25,50,75,np.inf]
        labels=['0 a <25 %','25 a <50 %','50 a <75 %','75 a 100 %']; unit='% de population a plus de 400 m d un arret publie'
    else:
        colors=['#edb46d','#b6cdb0','#60a5a0','#087f70']; bounds=[25,50,75,np.inf]
        labels=['0 a <25 %','25 a <50 %','50 a <75 %','75 a 100 %']; unit='% de population proche d une station connue' if cells is None else '% : carreau proche d une station connue'
    points=folium.FeatureGroup(name=title,show=True).add_to(m)
    # Volumes pour rendre les grandes cibles visibles, jamais un bonus rural.
    shown=valid.sort_values('map_order',ascending=False,na_position='last').head(600)
    shown=shown.sort_values('map_value',ascending=True,kind='stable')
    for r in shown.itertuples():
        color=next(c for b,c in zip(bounds,colors) if r.map_value<b)
        name=str(getattr(r,'commune',getattr(r,'commune_code','Carreau')))
        identifier=str(getattr(r,'commune_code',getattr(r,'cell_id','')))
        value=f'{r.map_value:,.0f}' if theme=='target' else f'{r.map_value:.1f}'
        popup=(f'<b>{html.escape(name)}</b><br>{html.escape(identifier)}<br><b>{value} {html.escape(unit)}</b><br>'
               f'{html.escape(description)}<br>Indicateur descriptif ; aucun changement des poids ou du score.')
        marker=folium.CircleMarker([r.latitude,r.longitude],radius=5,color=color,fill=True,fill_color=color,
                                   fill_opacity=.85,weight=1,tooltip=f'{name} · {value}',popup=folium.Popup(popup,max_width=360))
        marker.options['pane']='priority_points'; marker.add_to(points)
    missing=folium.FeatureGroup(name='Indicateur manquant - pas zero',show=True).add_to(m)
    for r in data.loc[data.map_value.isna()].head(600).itertuples():
        folium.CircleMarker([r.latitude,r.longitude],radius=3,color='#77838c',fill=True,
                            tooltip=f"{getattr(r,'commune',getattr(r,'commune_code','Carreau'))} : indicateur manquant").add_to(missing)
    stations=france['stations']
    if len(data) and (cells is not None or len(ranked)<5000):
        stations=stations.loc[stations.latitude.between(data.latitude.min()-.1,data.latitude.max()+.1)&stations.longitude.between(data.longitude.min()-.1,data.longitude.max()+.1)]
    fit_study_bounds(m,data)
    add_known_stations(m,stations,show=theme=='citiz')
    folium.LayerControl(collapsed=True).add_to(m)
    swatches=''.join(f'<div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;background:{c};margin-right:7px"></span>{label}</div>' for c,label in zip(colors,labels))
    foot=('Chaleur bleu → jaune → rouge : concentration relative, echelle log ; accumulation des voisins. Points violets : volumes estimes. ' if theme=='target' else '')
    foot+=('Points : 600 plus grandes cibles estimees du perimetre. ' if theme=='target' else 'Points : 600 communes ou carreaux les plus peuples du perimetre. ')
    foot+='Couleur selon l indicateur ; filtrer pour explorer les autres communes. Gris : manquant.'
    m.get_root().html.add_child(folium.Element(f'<div id="{m.get_name()}_legend" style="position:absolute;bottom:76px;left:14px;z-index:999;background:white;padding:10px;border-radius:8px;box-shadow:0 2px 8px #0003;font:11px Arial;line-height:1.6;max-width:230px"><details><summary style="cursor:pointer"><b>Legende · {html.escape(title)}</b></summary>{html.escape(unit)}{swatches}<div style="font-size:11px;color:#52606b">{foot}</div></details></div>'))
    _map_travel_controls(m)
    return m


def france_map(france, ranked, selected, cells=None, evidence=False,theme='priority'):
    if evidence: theme='evidence'  # compatibilite des anciens appels
    if theme not in MAP_THEMES: raise ValueError('Carte inconnue : '+str(theme))
    if theme in ['target','transit','citiz']: return _indicator_map(france,ranked,cells,theme)
    evidence=theme=='evidence'
    import folium
    from folium.plugins import HeatMap, FastMarkerCluster, Fullscreen
    m=folium.Map(location=[46.7,2.3],zoom_start=5,zoom_snap=.1,tiles='OpenStreetMap',height=540,
                 control_scale=True,prefer_canvas=True)
    m.get_root().height='540px'
    Fullscreen(position='topleft').add_to(m)
    # Les marqueurs de score doivent rester au-dessus du canvas de chaleur.
    folium.map.CustomPane('priority_points',z_index=625,pointer_events=True).add_to(m)
    national=cells is None
    frame=ranked.dropna(subset=['latitude','longitude'])
    heat=folium.FeatureGroup(name='Chaleur des priorites (communes)' if national else 'Besoin potentiel (carreaux 200 m)',show=not evidence).add_to(m)
    if national:
        values=frame.dropna(subset=['priority_score']).copy()
        target='uncovered_gap_licensed_adults_estimate' if 'uncovered_gap_licensed_adults_estimate' in values else 'uncovered_gap_no_car_proxy'
        weight=values[target].clip(lower=0)*values.priority_score/100
    else:
        values=cells.dropna(subset=['no_car_households_proxy']).copy()
        target='licensed_adults_no_car_estimate' if 'licensed_adults_no_car_estimate' in values else 'no_car_households_proxy'
        weight=values[target]*values.transit_gap_proxy*(~values.citiz_covered.fillna(True).astype(bool))
    intensity=heat_intensities(weight)
    HeatMap([[float(a),float(b),float(w)] for a,b,w in zip(values.latitude,values.longitude,intensity) if w>0],
            radius=16 if national else 14,blur=12,min_opacity=.12,max_zoom=5 if national else 9,
            gradient=HEAT_GRADIENT).add_to(heat)
    points=folium.FeatureGroup(name='Territoires a etudier' if national else 'Emplacements proposes - parking a verifier').add_to(m)
    chosen_ids=set(selected.commune_code if national else selected.candidate_id)
    detail_cols=['commune','priority_score','cost_total_eur','points_age','cost_penalty']
    display_frame=pd.concat([frame.loc[frame.priority_score.notna()].head(250),selected]).drop_duplicates('commune_code' if national else 'candidate_id')
    # En dezoom, les voisins se recouvrent : dessiner les fortes priorites en dernier.
    display_frame=display_frame.sort_values('priority_score',ascending=True,kind='stable')
    for r in display_frame.itertuples():
        identifier=r.commune_code if national else r.candidate_id
        chosen=identifier in chosen_ids
        if national:
            detail=(f"Commune {html.escape(r.commune_code)} · {html.escape(r.department_name)}<br>"
                    f"Menages sans voiture du RP2022 : {r.no_car_households_rp2022:,.0f}<br>"
                    f"Menages sans voiture hors couverture Citiz (proxy) : {r.uncovered_no_car_proxy:,.0f}<br>"
                    f"Cible permis B sans vehicule (estimation EMP2019) : {getattr(r,'uncovered_licensed_adults_no_car_estimate',np.nan):,.0f}<br>"
                    "Centre de commune : ni emplacement ni habitants gagnes par une station.")
        else:
            detail=(f"Habitants du departement nouvellement a moins de 400 m : {r.new_population:,.0f}<br>"
                    f"Menages sans voiture (proxy) : {r.new_no_car_households_proxy:,.0f}<br>"
                    f"Station Citiz connue la plus proche : {getattr(r,'nearest_existing_citiz_m',np.nan):,.0f} m<br>"
                    "Centre hors exclusion Citiz de 400 m (seuil heuristique).<br>"
                    "Centre de carreau ; distance aerienne ; parking a verifier.")
        cost_text=f'{r.cost_total_eur:,.0f} EUR' if np.isfinite(r.cost_total_eur) and getattr(r,'cost_source','') else 'non documente'
        penalty_text=f'{r.cost_penalty:.1f} points' if np.isfinite(r.cost_penalty) else 'non calculee'
        popup=(f"<b>{html.escape(str(r.commune))}</b><br>Score {r.priority_score:.1f}/100"
               f"<br>{detail}<br>Cout sur horizon documente : {cost_text}"
               f"<br>Reference cout : {html.escape(str(getattr(r,'cost_source','') or 'non renseignee'))}"
               f"<br>Penalite TC deja proches : {getattr(r,'transit_coverage_penalty',0):.1f} points"
               f"<br>Penalite cout : {penalty_text} · bonus 25–39 : {r.points_age:.1f}"
               f"<br>Solidite : {html.escape(str(getattr(r,'evidence_status','modele_non_valide')))}"
               f"<br>Part de carreaux imputes : {getattr(r,'imputed_cell_share',np.nan):.1%} (pas part de population)"
               "<br>Usage occasionnel, horaires TC et parking : a documenter. Permis : estimation d'enquete, pas registre local actuel.")
        fill_color='#7964aa' if evidence else priority_color(r.priority_score)
        marker=folium.CircleMarker([r.latitude,r.longitude],radius=9 if chosen else 6,
                            color='#087f70' if chosen else fill_color,weight=3 if chosen else 2,
                            fill=True,fill_color=fill_color,fill_opacity=.9,
                            tooltip=f'{r.commune} · {r.priority_score:.1f}',popup=folium.Popup(popup,max_width=390))
        marker.options['pane']='priority_points'
        marker.add_to(points)
    stations=france['stations']
    # Filtre uniquement visuel local: le moteur de couverture utilise toutes les stations.
    if len(frame) and (cells is not None or len(frame)<5000):
        extent=cells if cells is not None else frame
        stations=stations.loc[stations.latitude.between(extent.latitude.min()-.1,extent.latitude.max()+.1)&stations.longitude.between(extent.longitude.min()-.1,extent.longitude.max()+.1)]
    add_known_stations(m,stations,show=True)
    if national:
        missing=folium.FeatureGroup(name='Permis ou donnees fines manquants - pas de score',show=True).add_to(m)
        for r in frame.loc[frame.priority_score.isna()].itertuples():
            folium.CircleMarker([r.latitude,r.longitude],radius=3,color='#77838c',fill=True,
                                tooltip=f'{r.commune} : donnees insuffisantes',
                                popup=f"Commune conservee dans le perimetre. Permis : {html.escape(str(getattr(r,'license_status','non renseigne')))}. Grille : {html.escape(str(getattr(r,'grid_status','non renseignee')))}. Aucun besoin suppose nul.").add_to(missing)
    fit_study_bounds(m,cells if cells is not None else frame)
    folium.LayerControl(collapsed=True).add_to(m)
    swatches=('Violet : estimations et proxys, validation requise' if evidence else ''.join(f'<div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;background:{color};margin-right:7px"></span>{label}</div>' for _,color,label in PRIORITY_COLORS))
    uid=m.get_name()
    m.get_root().html.add_child(folium.Element(
        f'<div id="{uid}_legend" style="position:absolute;bottom:76px;left:14px;z-index:999;background:white;padding:10px;border-radius:8px;box-shadow:0 2px 8px #0003;font:11px Arial;line-height:1.6;max-width:230px"><details><summary style="cursor:pointer"><b>Legende de la carte</b></summary>'+
        ('<b>Solidite des donnees</b><br>' if evidence else '<b>Territoires : score de priorite /100</b>')+swatches+
        '<div style="color:#77838c">Gris : donnees insuffisantes</div>'
        '<div>Contour vert : retenu sous budget</div>'
        '<div>Carre C / groupe bleu : Citiz existant</div>'
        '<div style="font-size:11px">Chaleur bleu → jaune → rouge : concentration relative, echelle log, accumulation des voisins ; pas le score /100.</div>'
        '<div style="max-width:230px;font-size:11px;color:#52606b">Seuils fixes entre scenarios. Points = centres des communes ou carreaux ; top 250 scores calculables, zero inclus.</div></details></div>'))
    _map_travel_controls(m)
    return m


def territory_metrics(ranked,cells=None):
    """Agregats sur les populations connues, jamais sur les rayons qui se recouvrent."""
    data=ranked if cells is None else cells
    def numeric(name):
        return pd.to_numeric(data[name],errors='coerce') if name in data else pd.Series(np.nan,index=data.index)
    pop=numeric('population')
    if cells is None:
        far=numeric('population_far_stops')
        covered=pop-numeric('uncovered_population')
        target=numeric('uncovered_licensed_adults_no_car_estimate')
    else:
        distance=numeric('nearest_stop_m')
        far=pop.where(distance.gt(400),0).where(distance.notna())
        flag=data.citiz_covered if 'citiz_covered' in data else pd.Series(pd.NA,index=data.index)
        covered=pop.where(flag.eq(True).fillna(False),0).where(flag.notna())
        target=numeric('licensed_adults_no_car_estimate').where(flag.eq(False).fillna(False),0).where(flag.notna())
    def share(values):
        known=pop.notna()&values.notna()
        denominator=pop.where(known).sum(min_count=1)
        numerator=values.where(known).sum(min_count=1)
        return (100*numerator/denominator if np.isfinite(denominator) and denominator>0 else np.nan,numerator)
    far_share,far_population=share(far)
    citiz_share,citiz_population=share(covered)
    return dict(target=target.sum(min_count=1),population=pop.sum(min_count=1),far_share=far_share,
                far_population=far_population,citiz_share=citiz_share,citiz_population=citiz_population)


UI_STYLE='''<style>
.cb,.cb-report{font:13px "Segoe UI",Arial,sans-serif;color:#203d43;background:#f6f8f9;padding:16px;border:1px solid #e6ecee;border-radius:10px;box-sizing:border-box;white-space:normal}
.cb *,.cb-report *{box-sizing:border-box}.cb h3,.cb h4,.cb p{margin:0 0 7px}.cb h3{font-size:17px;font-weight:600}.cb h4{font-size:12px;font-weight:600}.cb small{display:block;font-size:11px;color:#657b80;line-height:1.55}
.cb-head{display:flex;justify-content:space-between;align-items:center;gap:12px;margin-bottom:14px}.cb-head small{font-size:11px}.cb-cards{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px;margin-bottom:14px}.cb-card{background:white;border:1px solid #e1e8eb;border-radius:8px;padding:13px 14px;border-top:2px solid #168487}.cb-label{font-size:10px;font-weight:600;color:#61797e}.cb-card strong{display:block;font-size:28px;line-height:1.45;font-weight:600;letter-spacing:-.6px;color:#163f48}
.cb-body{display:grid;grid-template-columns:minmax(0,1fr) 230px;gap:14px}.cb-map{background:white;border:1px solid #e1e8eb;border-radius:8px;overflow:hidden;min-width:0}.cb-maphead{padding:12px 14px;border-bottom:1px solid #edf1f2}.cb-maphead p{font-size:11px;color:#657b80;margin:0;line-height:1.5}.cb-map iframe{display:block;border:0!important}
.cb-side{display:flex;flex-direction:column;gap:12px;max-height:610px;overflow:auto}.cb-side section{background:white;border:1px solid #e1e8eb;border-radius:8px;padding:13px}.cb-side h4{margin-top:2px}.cb-place{display:flex;justify-content:space-between;gap:8px;align-items:center;padding:10px 0;border-top:1px solid #edf1f2}.cb-place>b{font-size:20px;font-weight:600}.cb-place span{min-width:0}.cb-place small{overflow-wrap:anywhere;font-size:10px}.cb-swatches{display:flex;flex-wrap:wrap;gap:6px 14px;margin:10px 0 6px}.cb-swatches span{font-size:10px;color:#61797e}.cb-swatches i{display:inline-block;width:8px;height:8px;border-radius:50%;margin-right:5px}.cb-rule{border-left:2px solid #168487!important}.cb-rule b{display:block;margin-bottom:5px;font-size:12px}
.cb-term{display:grid;grid-template-columns:1fr auto;gap:4px;font-size:10px;margin:9px 0}.cb-term>div{grid-column:1/-1;background:#edf2f3;height:5px;border-radius:4px}.cb-term i{display:block;height:5px;background:#218b91;border-radius:4px}.cb-footer{font-size:10px;color:#71858a;margin-top:12px}.cb facts{display:block}.cb-fact{display:flex;justify-content:space-between;gap:8px;padding:7px 0;border-bottom:1px solid #eef2f3;font-size:11px}.cb-fact b{font-weight:500;text-align:right}.cb summary,.cb-report summary{cursor:pointer;font-size:11px;font-weight:600;color:#305b64;padding:4px 0}.cb details p{font-size:11px;line-height:1.65;margin:8px 0;color:#657b80}
.cb-report{overflow-x:auto;line-height:1.65}.cb-report h3{font-size:16px;font-weight:600;margin:15px 0 10px}.cb-report h4{font-size:13px}.cb-report p,.cb-report details{color:#5e747a;font-size:12px}.cb-report table{border-collapse:collapse;width:100%;font-size:11px;border:none;background:white;margin:12px 0}.cb-report th{background:#edf3f4;color:#24484f;font-weight:600;text-align:left;position:sticky;top:0}.cb-report th,.cb-report td{padding:9px 11px;border:0;border-bottom:1px solid #e4ebee;vertical-align:top}.cb-report tr:nth-child(even){background:#f8fafb}.cb-report a{color:#087f80}
.cb-analysis-table{background:white;border:1px solid #e1e8eb;border-radius:8px;padding:12px 14px;margin:12px 0}.cb-analysis-table summary{font-size:13px!important}.cb-analysis-table summary span{font-size:10px;color:#71858a;font-weight:400;margin-left:12px}.cb-table-scroll{max-height:500px;overflow:auto}.cb-table-scroll th{position:static;white-space:normal;min-width:100px}.cb-table-scroll td{white-space:nowrap}.cb-table-scroll thead tr:first-child th{border-bottom:2px solid #cbdcde}
@media(max-width:900px){.cb-body{grid-template-columns:1fr}.cb-side{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));max-height:none}.cb-cards{gap:8px}.cb-card{padding:10px}.cb-card strong{font-size:24px}}
@media(max-width:540px){.cb-cards{grid-template-columns:1fr}.cb-side{grid-template-columns:1fr}.cb-head{align-items:flex-start}}
</style>'''


def report_panel(content):
    return UI_STYLE+'<div class="cb-report">'+content+'</div>'


def dashboard_html(france,ranked,selected,m,cells,theme,scope_label,expert=False):
    """Habillage BI commun : indicateurs, carte centrale et lecture contextuelle."""
    esc=lambda text:html.escape(str(text))
    number=lambda value:f'{value:,.0f}'.replace(',',' ') if np.isfinite(value) else 'Non disponible'
    title,description=MAP_THEMES[theme]
    valid=ranked.loc[ranked.priority_score.notna()]
    availability=map_availability(ranked,cells,theme)
    metrics=territory_metrics(ranked,cells)
    def card(label,value,sub):
        return f'<div class="cb-card"><div class="cb-label">{esc(label)}</div><strong>{esc(value)}</strong><small>{esc(sub)}</small></div>'
    percent=lambda value:f'{value:.1f} %' if np.isfinite(value) else 'Non disponible'
    cards=card('Permis B, sans vehicule',number(metrics['target']),'Estimation hors couverture Citiz')
    cards+=card('Eloignes des arrets',percent(metrics['far_share']),f'{number(metrics["far_population"])} habitants a plus de 400 m')
    cards+=card('Proches de Citiz',percent(metrics['citiz_share']),f'{number(metrics["citiz_population"])} habitants a moins de 400 m')
    top=valid.sort_values('priority_score',ascending=False).head(3)
    entries=''.join(f'<div class="cb-place"><span><b>{esc(r.commune)}</b><small>{esc(getattr(r,"candidate_id","")) if expert else "Carreau a verifier" if cells is not None else esc(getattr(r,"department_name","Commune"))}</small></span><b style="color:{priority_color(r.priority_score)}">{r.priority_score:.1f}</b></div>' for r in top.itertuples())
    if not entries: entries='<p>Aucun score calculable ; absence de donnees ne signifie pas absence de besoin.</p>'
    extras=''
    if expert:
        if len(top):
            first=top.iloc[0]
            bars=''
            for label,keys in [('Cible permis',['points_no_car']),('Manque TC',['points_transit','points_transit_gap']),('Population',['points_population','points_new_population']),('Social',['points_equity']),('Age 25–39',['points_age'])]:
                key=next((key for key in keys if key in first.index),None)
                value=first[key] if key else np.nan
                if np.isfinite(value): bars+=f'<div class="cb-term"><span>{esc(label)}</span><b>{value:.1f} pts</b><div><i style="width:{min(max(value,0),100)}%"></i></div></div>'
            extras+=f'<section><h4>Decomposition · {esc(first.commune)}</h4>{bars}<small>Avant penalites TC ({first.get("transit_coverage_penalty",0):.1f} pts) et couts documentes.</small></section>'
        extras+=f'<section><h4>Qualite et couverture</h4><p>{number(len(valid))} scores calculables / {number(len(ranked))}<br>{number(ranked.priority_score.isna().sum())} scores absents</p><small>Inventaire Citiz partiel possible. Proximite TC sans horaires. Estimation locale du permis, pas registre actuel.</small></section>'
    if theme=='priority':
        swatches=''.join(f'<span><i style="background:{color}"></i>{esc(label)}</span>' for _,color,label in PRIORITY_COLORS)
        reading='Points : score /100. Chaleur : concentration relative bleu → jaune → rouge.'
    elif theme=='evidence':
        swatches='<span><i style="background:#7964aa"></i>Modele a valider</span><span><i style="background:#77838c"></i>Donnees absentes</span>'
        reading='Couleur : statut des donnees, pas une probabilite de precision.'
    elif theme=='target':
        swatches='<span><i style="background:#b09bd8"></i>Cible estimee</span><span><i style="background:#77838c"></i>Valeur absente</span>'
        reading='Points violets : volume estime, voir les seuils dans Legende. Chaleur : concentration relative bleu → jaune → rouge.'
    else:
        colors=['#2878a5','#e3c35b','#ed9046','#b52b42'] if theme=='transit' else ['#edb46d','#b6cdb0','#60a5a0','#087f70']
        swatches=''.join(f'<span><i style="background:{color}"></i>{label}</span>' for color,label in zip(colors,['<25 %','25 a <50 %','50 a <75 %','75 a 100 %']))
        reading='Couleur : part de population eloignee des arrets.' if theme=='transit' else 'Couleur : part de population proche du Citiz connu.'
    facts=(f'<div class="cb-fact"><span>Population etudiee</span><b>{number(metrics["population"])}</b></div>'
           '<div class="cb-fact"><span>Distance aux arrets</span><b>A vol d’oiseau</b></div>'
           '<div class="cb-fact"><span>Horaires / frequences</span><b>Non renseignes</b></div>'
           '<div class="cb-fact"><span>Disponibilite Citiz</span><b>Non mesuree</b></div>')
    sources=('<details><summary>Sources et limites</summary><p>Population : Filosofi2021. Motorisation : RP2022. Permis : EMP2019, repartition locale estimee. '
             'Arrets : inventaire PAN, couverture possiblement incomplete. Stations : flux GBFS Citiz, inventaire possiblement incomplet. '
             'Les seuils de 400 m sont a vol d’oiseau. Aucun budget ou cout suppose.</p></details>')
    side=(f'<section><h4>Priorites a examiner <small>Score de besoin /100</small></h4>{entries}'
          '<small>Stationnement et cheminements a verifier.</small></section>'+extras+
          f'<section><h4>Transports et donnees</h4>{facts}{sources}</section>'+
          ('<section class="cb-rule"><b>Implantation locale</b><small>Centres a 400 m ou moins des stations connues exclus. Seuil heuristique.</small></section>' if expert else ''))
    return (UI_STYLE+f'<div class="cb" data-view="{"expert" if expert else "simple"}"><div class="cb-head"><div><h3>{esc(scope_label)}</h3><small>{number(len(ranked))} {"centres candidats" if cells is not None else "communes"} · {number(availability["available"])} valeurs cartographiables</small></div></div>'
            f'<div class="cb-cards">{cards}</div><div class="cb-body"><div class="cb-map"><div class="cb-maphead"><h4>{esc(title)}</h4><p>{esc(description)}</p><div class="cb-swatches">{swatches}</div><small>{reading}</small></div>{m._repr_html_()}</div><aside class="cb-side">{side}</aside></div>'
            '<div class="cb-footer">Parts calculees sur la population renseignee. Cible permis estimee ; pas une prevision de clients. C : station Citiz connue.</div></div>')


def local_department(france, department, scenario, costs=None):
    """Recalcul du gain marginal sur les habitants DU departement uniquement."""
    if department not in STUDY_DEPARTMENTS: raise ValueError('Choisir un departement de France metropolitaine hors Corse.')
    # Dans le notebook, ces definitions viennent de la cellule moteur V1.
    try:
        from citiz_v1 import Config, rank_and_select
    except ImportError:
        Config=globals()['Config']; rank_and_select=globals()['rank_and_select']
    files=sorted((Path(france['workdir'])/'cells'/department).glob('*.csv'))
    if not files and 'restore_france_cache' in globals():
        restore_france_cache(france['workdir'],department=department)
        files=sorted((Path(france['workdir'])/'cells'/department).glob('*.csv'))
    if not files: raise ValueError('Pas de grille fine disponible pour ce departement. La commune reste visible dans la vue France.')
    data=pd.concat([pd.read_csv(p,dtype={'commune_code':str,'department':str,'region':str}) for p in files],ignore_index=True).drop_duplicates('cell_id')
    # Eviter de recalculer sur une copie partielle d'un departement cherche par nom.
    data=data.dropna(subset=['no_car_rate']).copy()
    if data.empty: raise ValueError('Motorisation absente : emplacements non calculables.')
    # Coordonnees cartesiennes locales: valables aussi en outre-mer et en Corse.
    transform=Transformer.from_crs(4326,f'+proj=aeqd +lat_0={data.latitude.mean()} +lon_0={data.longitude.mean()} +datum=WGS84 +units=m',always_xy=True)
    data['x'],data['y']=transform.transform(data.longitude.to_numpy(),data.latitude.to_numpy())
    data['transit_gap']=data.transit_gap_proxy
    # Recalcul avec l'inventaire courant, toutes communes et limites comprises.
    from scipy.spatial import cKDTree
    stations=france['stations'].dropna(subset=['latitude','longitude'])
    tree=cKDTree(sphere_xyz(stations.latitude,stations.longitude)) if len(stations) else None
    data['nearest_citiz_m']=nearest_metres(tree,data.latitude,data.longitude)
    data['citiz_covered']=data.nearest_citiz_m.le(400).where(data.nearest_citiz_m.notna())
    if 'overview' in france and 'license_coefficient' in france['overview']:
        coefficients=france['overview'].set_index('commune_code').license_coefficient
        data['licensed_adults_no_car_estimate']=data.no_car_households_proxy*data.commune_code.map(coefficients)
    common_cost=(scenario.site_cost_eur is not None and bool(scenario.cost_source.strip()))
    documented_overrides={code:value for code,value in (costs or {}).items() if isinstance(value,tuple) and str(value[1]).strip()}
    finance=(scenario.budget_eur is not None and scenario.budget_source.strip() and scenario.horizon_years is not None and (common_cost or bool(documented_overrides)))
    # Zero est un neutre algorithmique pour calculer le BESOIN, jamais un prix.
    cfg=Config(budget_eur=scenario.budget_eur if finance else 0,max_stations=scenario.max_sites if finance else 0,
               setup_eur=scenario.site_cost_eur if finance and common_cost else 0,vehicle_eur=0,annual_operation_eur=0,
               horizon_years=scenario.horizon_years if finance else 1,
               budget_source=scenario.budget_source if finance else '',cost_source=(scenario.cost_source or 'references_par_site') if finance else '',
               cost_reference_eur=scenario.budget_eur if finance and scenario.budget_eur>0 else 1,
               age_bonus_max=scenario.age_bonus,max_cost_penalty=scenario.cost_penalty_weight if finance else 0,
               no_car_weight=scenario.no_car_weight,transit_weight=scenario.transit_weight,
               population_weight=scenario.population_weight,equity_weight=scenario.equity_weight,
               transit_coverage_penalty=scenario.transit_coverage_penalty)
    raw_costs={} if common_cost else {code:np.nan for code in data.cell_id}
    for code,value in (costs or {}).items():
        raw_costs[code]=value[0] if isinstance(value,tuple) and str(value[1]).strip() else np.nan
    ranked,selected,_=rank_and_select(data,cfg,raw_costs if finance else None)
    def source_for(code):
        if not finance: return ''
        if code in (costs or {}): return str(documented_overrides.get(code,('',''))[1]).strip()
        return scenario.cost_source.strip() if common_cost else ''
    ranked['cost_source']=ranked.candidate_id.map(source_for)
    selected['cost_source']=selected.candidate_id.map(source_for)
    ranked['budget_source']=scenario.budget_source.strip()
    selected['budget_source']=scenario.budget_source.strip()
    if not finance:
        for column in ['cost_total_eur','cost_penalty','cost_per_new_resident_eur']:
            ranked[column]=np.nan
        ranked['cost_status']='non_documente'
        ranked['score_status']='besoin_hors_couts_budget_non_documente'
    if 'annotate_evidence' in globals():
        ranked=annotate_evidence(ranked); selected=annotate_evidence(selected)
    return ranked,selected,data


def show_france_interactive(france,render=True):
    import ipywidgets as w
    from IPython.display import display as ipython_display, HTML, update_display
    # Fabric fournit son propre display pour enregistrer les widgets interactifs.
    # Le rendu HTML de carte conserve le display IPython et son display_id.
    widget_display=globals().get('display',ipython_display)
    import uuid
    base=france['overview'].loc[france['overview'].region.isin(STUDY_REGIONS)].copy()
    region_options=[('Metropole hors Corse','')]+sorted({(r.region_name,str(r.region)) for r in base.itertuples() if pd.notna(r.region)})
    field_layout=lambda:w.Layout(width='32%',min_width='280px',flex='1 1 280px',margin='0 8px 8px 0',height='28px')
    field_style={'description_width':'76px'}
    region=w.Dropdown(options=region_options,description='Region',layout=field_layout(),style=field_style)
    department=w.Dropdown(options=[('Tous les departements','')],description='Departement',layout=field_layout(),style=field_style)
    search=w.Text(description='Commune',placeholder='Nom ou code INSEE',continuous_update=True,layout=field_layout(),style=field_style)
    mode=w.Dropdown(options=[('Communes','national')],value='national',description='Echelle',
                    layout=field_layout(),style=field_style)
    presentation=w.ToggleButtons(options=[('Vue simple','simple'),('Analyse complete','expert')],value='simple',
                                 style={'button_width':'145px'},layout=w.Layout(width='310px'))
    view_simple=w.Button(description='Vue simple',layout=w.Layout(width='140px'))
    view_expert=w.Button(description='Analyse complete',layout=w.Layout(width='155px'))
    view_simple.on_click(lambda _:setattr(presentation,'value','simple'))
    view_expert.on_click(lambda _:setattr(presentation,'value','expert'))
    view_switch=w.HBox([view_simple,view_expert],layout=w.Layout(width='310px'))
    map_view=w.Dropdown(options=[(label,key) for key,(label,_) in MAP_THEMES.items()],value='priority',description='Carte',layout=field_layout(),style=field_style)
    budget=w.Text(value='',placeholder='Non renseigne',description='Budget EUR',continuous_update=True)
    cost=w.Text(value='',placeholder='Non renseigne',description='Cout/site EUR',continuous_update=True)
    budget_source=w.Text(value='',placeholder='Reference du budget valide',description='Source budget',continuous_update=True,layout=w.Layout(width='600px'))
    cost_source=w.Text(value='',placeholder='Devis, tarif ou reference + horizon commun',description='Source cout',continuous_update=True,layout=w.Layout(width='600px'))
    horizon=w.Text(value='',placeholder='Non renseigne',description='Horizon (ans)',continuous_update=True)
    max_sites=w.IntSlider(value=20,min=0,max=100,step=1,description='Max sites',continuous_update=False)
    weights=[w.IntSlider(value=value,min=0,max=100,description=label,continuous_update=False,
                         style={'description_width':'150px'},layout=w.Layout(width='350px'))
             for label,value in [('Permis + sans voiture',35),('Eloignement TC',35),('Population',20),('Contexte social',5)]]
    age=w.IntSlider(value=5,min=0,max=5,description='Bonus 25–39',continuous_update=False)
    penalty=w.IntSlider(value=25,min=0,max=50,description='Penalite cout',continuous_update=False)
    tc_penalty=w.IntSlider(value=35,min=0,max=60,description='TC deja proches',continuous_update=False,
                          style={'description_width':'150px'},layout=w.Layout(width='350px'))
    apply=w.Button(description='Appliquer',layout=w.Layout(width='112px'))
    apply.add_class('citiz-primary')
    reset=w.Button(description='Reinitialiser',tooltip='Reinitialiser les filtres',layout=w.Layout(width='110px'))
    export=w.Button(description='Exporter CSV',layout=w.Layout(width='128px'))
    status=w.HTML(); kpis=w.HTML(); canvas=w.HTML(); table=w.HTML(); analysis=w.HTML(); usages=w.HTML()
    state={'busy':False,'last':None,'cells':None,'maps':{},'analysis_tables':{},'content_html':'','map_id':'citiz_france_'+uuid.uuid4().hex,'map_published':False,'ranking_runs':0}
    header=w.HTML("""<div style='font:12px Segoe UI;color:#61797e;padding:5px 4px 14px'>
    <b style='font-size:23px;font-weight:600;color:#17464e'>Citiz</b><span style='margin-left:13px'>Territoires &amp; mobilite</span></div>""",layout=w.Layout(flex='1 1 230px'))
    note="""<details><summary>Definitions, sources et limites du modele</summary><p>France metropolitaine : communes a etudier, pas stations validees.
    Cible permis B sans vehicule estimee avec EMP2019 et RP2022 ; pas registre local.
    Le score emploie la proximite des arrets publies, sans horaires, et une penalite de couverture ajustable. La carte Arrets montre separement la part de population eloignee de plus de 400 m.
    Pas de bonus rural. Bonus experimental 25-39 ans plafonne a 5 points. Les quatre poids principaux sont normalises sur 95 points.
    Sans budget, couts, horizon commun et references : selection financiere desactivee. Aucun montant suppose.
    Detail 200 m : centres de carreaux, distances aeriennes, parking et cheminement a verifier.</p></details>"""
    def render_content():
        if state['last'] is None:
            content='<p>Choisir un perimetre puis recalculer.</p>'
        elif tabs.value==0:
            theme=map_view.value
            scenario,ranked,selected,_,local=state['last']
            m=state['maps'].get(theme)
            if m is None:
                scenario,ranked,selected,_,local=state['last']
                m=france_map(france,ranked,selected,state['cells'],theme=theme)
                state['maps'][theme]=m
            last=list(state['last']); last[3]=m; state['last']=tuple(last)
            title,description=MAP_THEMES[theme]
            availability=map_availability(ranked,state['cells'],theme)
            scope_label=' · '.join(str(x) for x in [region.label,department.label if department.value else '',search.value.strip()] if x)
            if availability['available']==0:
                content=(f'<div style="max-width:950px;padding:20px;background:#fff4dd;border-left:5px solid #d9a52a;font:14px Segoe UI">'
                         f'<h3>Aucune valeur exploitable sur ce perimetre</h3><p>{html.escape(scope_label)} : {availability["total"]} lignes, '
                         f'{availability["geocoded"]} geolocalisees. Aucun score ou indicateur disponible pour cette carte.</p>'
                         '<p>Ce resultat ne signifie pas une absence de besoin. Choisir un autre territoire ou cliquer sur <b>Reinitialiser les filtres</b>.</p></div>')
            else:
                content=dashboard_html(france,ranked,selected,m,state['cells'],theme,scope_label,expert=presentation.value=='expert')
        elif tabs.value==2:
            content=report_panel(kpis.value+note+analysis.value)
        elif tabs.value==3:
            content=report_panel(usages.value)
        else:
            content=''
        state['content_html']=content
        if state['map_published']:
            update_display(HTML(content),display_id=state['map_id'])
    def view_changed(change):
        if state['busy']: return
        render_content()
    def update_scale():
        mode.options=[('Communes','national')]+([('Carreaux 200 m','local')] if department.value else [])
    def update_departments():
        subset=base if not region.value else base.loc[base.region.eq(region.value)]
        old=department.value
        options=[('Tous les departements','')]+sorted({(f'{r.department} · {r.department_name}',str(r.department)) for r in subset.itertuples() if pd.notna(r.department)})
        department.options=options
        if old in [code for _,code in options]: department.value=old
        update_scale()
    def params():
        raw=np.asarray([x.value for x in weights],dtype=float)
        normalized=raw*95/max(raw.sum(),1)
        def amount(text):
            value=text.strip().replace(' ','').replace(',','.')
            return None if not value else float(value)
        return FranceScenario(budget_eur=amount(budget.value),site_cost_eur=amount(cost.value),
                              budget_source=budget_source.value.strip(),cost_source=cost_source.value.strip(),max_sites=max_sites.value,
                              horizon_years=None if not horizon.value.strip() else int(horizon.value.strip()),
                              no_car_weight=normalized[0],transit_weight=normalized[1],population_weight=normalized[2],
                              equity_weight=normalized[3],age_bonus=age.value,cost_penalty_weight=penalty.value,
                              transit_coverage_penalty=tc_penalty.value,
                              region=region.value,department=department.value,search=search.value.strip())
    def recalc(*args):
        if state['busy']: return
        state['busy']=True; apply.disabled=True
        status.value='<p>Recalcul Python en cours…</p>'
        try:
            scenario=params()
            local=mode.value=='local'
            if local and not department.value: raise ValueError('Choisir un departement pour le detail 200 m.')
            overrides_file=Path(france['workdir'])/('site_cost_overrides.csv' if local else 'commune_cost_overrides.csv')
            costs=None
            if overrides_file.exists():
                overrides=pd.read_csv(overrides_file,dtype={'commune_code':str,'candidate_id':str})
                key='candidate_id' if local else 'commune_code'
                if overrides[key].duplicated().any(): raise ValueError('Couts dupliques pour un identifiant.')
                if 'source' not in overrides: raise ValueError('Ajouter la colonne source aux couts saisis.')
                costs={code:(amount,str(source).strip() if pd.notna(source) else '') for code,amount,source in zip(overrides[key],pd.to_numeric(overrides.cost_total_eur,errors='raise'),overrides.source)}
            if local:
                ranked,selected,cells=local_department(france,department.value,scenario,costs)
                if scenario.search:
                    status.value='<p>En detail, le calcul porte sur tout le departement. Vider Commune pour eviter une ambiguite de perimetre.</p>'
            else:
                ranked=france_rank(base,scenario,costs)
                selected=ranked.loc[ranked.selected]; cells=None
            state['ranking_runs']+=1
            m=france_map(france,ranked,selected,cells,theme=map_view.value)
            canvas.value='<p><b>Carte interactive ci-dessous :</b> zoom, couches, fiches au clic et plein ecran. Points colores par score avec seuils fixes. Chaleur relative au territoire affiche : cible estimee eloignee des TC et hors Citiz.</p>'
            total_cost=selected.cost_total_eur.sum()
            financed=(scenario.budget_eur is not None and scenario.budget_source and ranked.cost_penalty.notna().any())
            money=(f"<span><b>{total_cost:,.0f} EUR</b><br>cout des lignes retenues</span><span><b>{scenario.budget_eur-total_cost:,.0f} EUR</b><br>budget restant</span>" if financed else
                   '<span><b>Budget non documente</b><br>selection financiere desactivee</span><span><b>Couts non estimes</b><br>classement des besoins</span>')
            kpis.value=(f"<div style='display:flex;gap:28px;font:14px Segoe UI;padding:18px;background:#eff6f5;border-radius:8px'>"
                        f"<span><b style='font-size:24px'>{len(ranked):,}</b><br>{'candidats 200 m' if local else 'communes explorees'}</span>"
                        f"<span><b style='font-size:24px'>{len(selected) if financed else '—'}</b><br>{'emplacements retenus' if local else 'territoires retenus'}</span>"+money+'</div>')
            common=['commune','priority_score','evidence_status','transit_coverage_penalty','cost_total_eur','cost_source','points_age','cost_penalty']
            extra=['new_population','new_no_car_households_proxy','new_licensed_adults_no_car_estimate'] if local else ['department_name','uncovered_licensed_adults_no_car_estimate','license_status','grid_status']
            table.value=('<h3>Selection sous budget documente</h3>'+selected[common+extra].round(2).to_html(index=False,escape=True,na_rep='Non renseigne') if financed else
                         '<h3>Priorites de besoin — hors couts</h3><p>Selection financiere non calculee : fournir budget, couts et references.</p>'+ranked[common+extra].head(15).round(2).to_html(index=False,escape=True,na_rep='Non renseigne'))
            failures=france['sources'].loc[~france['sources'].status.eq('ok')&~france['sources'].status.str.startswith('inventaire')]
            state['analysis_tables']=build_analysis_tables(france,ranked,cells,territory_metrics)
            analysis.value=analysis_report(state['analysis_tables'],local)
            usages.value=mobility_panel_html(france,ranked) if 'mobility_panel_html' in globals() else '<p>Module usages non charge.</p>'
            state['last']=(scenario,ranked,selected,m,local)
            state['cells']=cells; state['maps']={map_view.value:m}
            render_content()
            status.value=''
        except Exception as error:
            state['last']=None
            status.value=f"<p style='color:#b43c30'>{html.escape(str(error))}</p>"
            kpis.value=''; canvas.value=''; table.value=''; analysis.value=''; usages.value=''
            state['maps']={}; state['cells']=None
            state['analysis_tables']={}
            render_content()
        finally:
            apply.disabled=False; state['busy']=False
    def region_changed(change):
        if state['busy']: return
        state['busy']=True; update_departments(); state['busy']=False; recalc()
    def reset_filters(*args):
        state['busy']=True
        region.value=''; update_departments(); department.value=''; search.value=''; mode.value='national'
        state['busy']=False; recalc()
    def save(*args):
        if state['last'] is None: return
        scenario,ranked,selected,m,local=state['last']
        stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
        folder=Path(france['workdir'])/'scenarios'/stamp; folder.mkdir(parents=True)
        ranked.to_csv(folder/'classement.csv',index=False,encoding='utf-8')
        selected.to_csv(folder/'selection.csv',index=False,encoding='utf-8')
        export_analysis_tables(state['analysis_tables'],folder)
        (folder/'parametres.json').write_text(json.dumps(asdict(scenario),indent=2),encoding='utf-8')
        m.save(str(folder/'carte.html'))
        (folder/'vue_cartographique.json').write_text(json.dumps({'carte':map_view.value,'presentation':presentation.value,'cartes_disponibles':MAP_THEMES},ensure_ascii=False,indent=2),encoding='utf-8')
        (folder/'tableau_de_bord.html').write_text(state['content_html'],encoding='utf-8')
        for theme,cached_map in state['maps'].items(): cached_map.save(str(folder/f'carte_{theme}.html'))
        if 'publish_france_files' in globals(): publish_france_files(france,only_scenario=folder)
        status.value=f'<p>Scenario CSV et carte sauvegardes : {html.escape(str(folder))}</p>'
    update_departments()
    region.observe(region_changed,names='value')
    def department_changed(change):
        if state['busy']: return
        state['busy']=True; update_scale(); state['busy']=False; recalc()
    department.observe(department_changed,names='value')
    for control in [mode,max_sites,*weights,age,penalty,tc_penalty]: control.observe(recalc,names='value')
    apply.on_click(recalc); export.on_click(save); reset.on_click(reset_filters)
    for control in [budget_source,cost_source]: control.layout.width='410px'
    settings=w.HBox([w.VBox([w.HTML('<b>Poids du besoin</b>'),*weights,tc_penalty,age]),
                     w.VBox([w.HTML('<b>Scenario financier · sources requises</b>'),budget,cost,horizon,max_sites,penalty,budget_source,cost_source])],
                    layout=w.Layout(flex_flow='row wrap'))
    actions=w.HBox([apply,reset],layout=field_layout())
    actions.layout.padding='0 0 0 84px'
    actions.layout.align_items='center'
    decision=w.HBox([map_view,mode,actions],layout=w.Layout(flex_flow='row wrap',width='100%',margin='0'))
    # Un seul panneau monte : certains rendus Fabric empilent les enfants de Tab.
    panels=[w.VBox([]),settings,w.VBox([]),w.VBox([])]
    tabs=w.ToggleButtons(options=[('Cartes',0),('Parametres',1),('Analyse',2),('Usages et sources',3)],value=0)
    nav_buttons=[]
    for label,index in tabs.options:
        button=w.Button(description=label,layout=w.Layout(width='150px'))
        button.on_click(lambda _,index=index:setattr(tabs,'value',index))
        nav_buttons.append(button)
    tab_bar=w.HBox(nav_buttons,layout=w.Layout(flex_flow='row wrap'))
    panel=w.VBox([panels[0]])
    def update_toolbar():
        decision.children=(map_view,mode,actions)
    def switch_panel(change):
        panel.children=(panels[change['new']],)
        update_view_buttons()
        update_toolbar()
        render_content()
    tabs.observe(switch_panel,names='value')
    map_view.observe(view_changed,names='value')
    expert_nav=w.VBox([])
    def presentation_changed(change):
        tabs.value=0
        update_view_buttons()
        update_toolbar()
        expert_nav.children=(w.HBox([tab_bar,export],layout=w.Layout(flex_flow='row wrap',align_items='center')),) if change['new']=='expert' else ()
        render_content()
    presentation.observe(presentation_changed,names='value')
    def update_view_buttons():
        for button,key in [(view_simple,'simple'),(view_expert,'expert')]:
            active=presentation.value==key
            button.style.button_color='#176f74' if active else '#f4f7f8'
            button.style.text_color='white' if active else '#31545c'
        for index,button in enumerate(nav_buttons):
            button.style.button_color='#176f74' if tabs.value==index else '#f4f7f8'
            button.style.text_color='white' if tabs.value==index else '#31545c'
    for button in [apply,reset,export,view_simple,view_expert,*nav_buttons]:
        button.style.font_family='Segoe UI'
        button.style.font_size='12px'
        button.style.font_weight='500'
        button.layout.border='1px solid #dce5e8'
        button.layout.height='28px'
        # Fabric conserve une hauteur de ligne native de 28 px : le centrage
        # flex evite que bordure et padding repoussent le texte vers le bas.
        button.layout.display='flex'
        button.layout.align_items='center'
        button.layout.justify_content='center'
        button.layout.padding='0'
        button.layout.margin='0 4px 0 0'
    apply.style.button_color='#176f74';apply.style.text_color='white'
    reset.style.button_color='#f4f7f8';export.style.button_color='#f4f7f8'
    reset.style.text_color='#31545c';export.style.text_color='#31545c'
    update_view_buttons()
    dashboard=w.VBox([w.HBox([header,view_switch],layout=w.Layout(flex_flow='row wrap',align_items='center')),
                       w.HBox([region,department,search],layout=w.Layout(flex_flow='row wrap')),
                       decision,expert_nav,panel,status],layout=w.Layout(width='100%'))
    dashboard.add_class('citiz-ui')
    recalc()
    if render:
        widget_display(dashboard)
        if state['last'] is not None:
            ipython_display(HTML(state['content_html']),display_id=state['map_id'])
            state['map_published']=True
    # Handles conserves pour verification et reprise sans recharger les sources.
    return dict(widget=dashboard,state=state,region=region,department=department,search=search,
                mode=mode,budget=budget,cost=cost,budget_source=budget_source,cost_source=cost_source,
                horizon=horizon,weights=weights,age=age,penalty=penalty,map_view=map_view,tabs=tabs,presentation=presentation,expert_nav=expert_nav,recalc=recalc,reset_filters=reset_filters)


In [ ]:
fr_dashboard=show_france_interactive(france,render=False)
display(fr_dashboard["widget"])
from IPython.display import display as _html_display, HTML
_html_display(HTML(fr_dashboard["state"]["content_html"]),display_id=fr_dashboard["state"]["map_id"])
fr_dashboard["state"]["map_published"]=True


In [ ]:
with diagnostic_output(france["workdir"],"sources"):
    archive_research_sources(france["workdir"])


In [ ]:
with diagnostic_output(france["workdir"],"publication"):
    # Dans Fabric : sauvegarder CSV, cartes et grilles dans OneLake
    _scenario,_ranked,_selected,_map,_local=fr_dashboard["state"]["last"]
    for _theme in MAP_THEMES:
        _variant=france_map(france,_ranked,_selected,fr_dashboard["state"]["cells"],theme=_theme)
        _filename="carte_france.html" if _theme=="priority" else "carte_france_"+_theme+".html"
        _variant.save(str(Path(france["workdir"])/"output"/_filename))
    publication_csv=publish_france_files(france)
